# 12 · GANs and Diffusion Models

**Part V — Generative models** · Estimated time: ~150 min

Notebook 11 · Autoencoders and Variational Autoencoders ended with a generator whose samples were plausible but blurry. This notebook is about the two families that produce *sharp* samples: **generative adversarial networks**, which replace the likelihood by a learned critic, and **diffusion models**, which turn generation into a long sequence of denoising steps. Both are derived carefully — the optimal discriminator and the Jensen–Shannon divergence for GANs; the forward-process closed form, the reverse process and the $\epsilon$-prediction objective for diffusion — and both are implemented and trained on MNIST. Along the way we meet the Wasserstein GAN with gradient penalty, deterministic DDIM sampling, classifier-free guidance, and the flow-matching view that underlies the most recent generators. We close by comparing every generative family met in Part V.

## Learning objectives

- Place VAEs, GANs, diffusion and flow models in a taxonomy of generative models and explain what each optimizes.
- Derive the optimal discriminator of the GAN game and show that the generator then minimizes the Jensen–Shannon divergence; explain the non-saturating loss.
- Implement and train a DCGAN and a conditional GAN on MNIST, read their (uninformative) loss curves, and recognize mode collapse and discriminator dominance.
- Explain why the Jensen–Shannon divergence fails on disjoint supports, define the Wasserstein-1 distance, and implement WGAN-GP.
- Derive the closed form of the diffusion forward process, the Gaussian reverse process and the simplified $\epsilon$-prediction objective from the diffusion ELBO.
- Implement a time-conditioned U-Net, train a DDPM, sample with the ancestral and the DDIM samplers, and visualize the denoising trajectory.
- Implement classifier-free guidance and explain the fidelity–diversity trade-off it controls.
- Write out the flow-matching objective and train a 2-D flow-matching model; compare VAEs, GANs, diffusion and flows on quality, likelihood, speed and stability.

## Prerequisites

- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (Adam and its momentum parameters, normalization layers, reading loss curves)
- Notebook 06 · Convolutional Neural Networks (strided and transposed convolutions, GroupNorm/BatchNorm)
- Notebook 09 · Attention and Transformers (self-attention, used once inside the U-Net)
- Notebook 11 · Autoencoders and Variational Autoencoders (the ELBO, KL divergences, evaluating generative models)

## Contents

1. The generative modelling landscape
2. GAN theory
3. DCGAN on MNIST
4. Wasserstein GAN
5. Diffusion models: the theory
6. DDPM on MNIST
7. The modern landscape and flow matching
8. Comparison of generative model families

followed by Exercises, Summary and Further reading.

In [ ]:
import copy, math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.utils import make_grid
from sklearn.datasets import make_swiss_roll

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Budgets used throughout the notebook (every training loop reads from these).
N_TRAIN = 8_000 if FAST_RUN else 60_000       # MNIST images used for the GANs
N_TRAIN_DIFF = 4_000 if FAST_RUN else 60_000  # MNIST images used for the diffusion models
EPOCHS_GAN = 2 if FAST_RUN else 20            # DCGAN / conditional GAN
EPOCHS_WGAN = 1 if FAST_RUN else 20           # WGAN-GP (each epoch does 5 critic updates per generator update)
EPOCHS_DIFF = 1 if FAST_RUN else 20           # DDPM
WIDTH_GAN = 32 if FAST_RUN else 64            # channel width of the DCGAN
WIDTH_UNET = 16 if FAST_RUN else 64           # base channel width of the U-Net
T_DIFF = 200 if FAST_RUN else 1000            # number of diffusion steps
BATCH_GAN, BATCH_DIFF = 64, 64
N_SAMPLE = 16 if FAST_RUN else 64             # images per sample grid
Z_DIM = 100
print(f"FAST_RUN={FAST_RUN}: N_TRAIN={N_TRAIN}, EPOCHS_GAN={EPOCHS_GAN}, EPOCHS_DIFF={EPOCHS_DIFF}, "
      f"WIDTH_GAN={WIDTH_GAN}, WIDTH_UNET={WIDTH_UNET}, T_DIFF={T_DIFF}")

### Data

As in Notebook 11 · Autoencoders and Variational Autoencoders we load MNIST once into an in-memory tensor and slice mini-batches from it directly. One difference: GANs with a `tanh` output layer and diffusion models with Gaussian noise both work with images scaled to $[-1, 1]$ rather than $[0, 1]$, so the pixels are rescaled here and mapped back with $(x + 1) / 2$ for display.

In [ ]:
def load_split(train):
    """MNIST as float tensors: X of shape (N, 1, 28, 28) scaled to [-1, 1], integer labels y of shape (N,)."""
    ds = datasets.MNIST(DATA_DIR, train=train, download=True)
    X = ds.data.unsqueeze(1).float().div(255.0) * 2.0 - 1.0
    return X, ds.targets.clone()

t0 = time.time()
X_train_full, y_train_full = load_split(train=True)
X_test, y_test = load_split(train=False)
print(f"loaded MNIST train {tuple(X_train_full.shape)}, test {tuple(X_test.shape)} in {time.time() - t0:.1f}s")

subset = torch.randperm(X_train_full.shape[0])
X_train = X_train_full[subset[:N_TRAIN]].to(device)
y_train = y_train_full[subset[:N_TRAIN]].to(device)
X_diff = X_train_full[subset[:N_TRAIN_DIFF]].to(device)
y_diff = y_train_full[subset[:N_TRAIN_DIFF]].to(device)
X_test, y_test = X_test.to(device), y_test.to(device)
print(f"X_train {tuple(X_train.shape)} on {X_train.device}, pixel range "
      f"[{X_train.min().item():.0f}, {X_train.max().item():.0f}]")

def iterate_batches(tensors, batch_size, shuffle=True):
    """Yield tuples of aligned mini-batches sliced from tensors that already live on the device."""
    n = tensors[0].shape[0]
    dev = tensors[0].device
    idx = torch.randperm(n, device=dev) if shuffle else torch.arange(n, device=dev)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield tuple(t[j] for t in tensors)

def show_images(x, nrow=8, title=None, figsize=(6, 6)):
    """Display a batch of images (N, 1, 28, 28) with values in [-1, 1] as a grid."""
    x = ((x.detach().float().cpu() + 1.0) / 2.0).clamp(0, 1)
    grid = make_grid(x, nrow=nrow, padding=1, pad_value=0.5)
    plt.figure(figsize=figsize)
    plt.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.tight_layout(); plt.show()

show_images(X_train[:16], nrow=8, title="Sixteen training digits", figsize=(6, 2))

## 1. The generative modelling landscape

A **generative model** is a model of the data distribution $p_{\text{data}}(x)$ from which we can draw new samples. Everything in Part V fits this one-line definition, yet the families differ profoundly in *what they optimize* and *what they give you*. It helps to sort them by their relationship to the likelihood.

**Likelihood-based models** define an explicit density $p_\theta(x)$ (or a bound on it) and fit it by maximum likelihood. Autoregressive models (the language models of Notebook 09 · Attention and Transformers, PixelCNN for images) factorize $p_\theta(x) = \prod_i p_\theta(x_i \mid x_{<i})$ and have an exact likelihood but sample one element at a time. Normalizing flows compose invertible layers and get an exact likelihood through the change-of-variables formula, at the price of architectural constraints. VAEs (Notebook 11 · Autoencoders and Variational Autoencoders) optimize a lower bound. Diffusion models, as we will see, also optimize a bound — a very long one, with one term per noise level — although in practice they are trained with a simplified weighted version of it.

**Implicit models** never write down a density. A GAN defines its distribution only through the sampling procedure "draw $z$, apply $G$", and learns by making a discriminator unable to tell samples from data. There is no likelihood to evaluate, which complicates evaluation and rules out density-based applications such as anomaly scoring, but there is also no likelihood to *compromise*: nothing pushes the generator to spread probability over blurry averages, which is one reason GAN samples are sharp.

**Score-based models** learn neither the density nor a sampler directly but the **score** $\nabla_x \log p(x)$, the direction in which the density increases. Sampling then follows the score with Langevin dynamics or an equivalent differential equation. Diffusion models are score-based models in disguise (Section 5.5), and flow-matching models (Section 7) learn a closely related velocity field.

| Family | Density | Training signal | Sampling cost | Typical strengths | Typical weaknesses |
|---|---|---|---|---|---|
| Autoregressive | exact | max. likelihood | $D$ sequential steps | exact likelihood, discrete data | slow sampling, no latent space |
| Normalizing flow | exact | max. likelihood | one pass | exact likelihood, invertible | constrained architectures |
| VAE | lower bound | ELBO | one pass | fast, latent space, stable | blurry samples |
| GAN | none | adversarial game | one pass | sharp samples, fast | unstable, mode collapse, no likelihood |
| Diffusion / score | bound | denoising regression | 10–1000 passes | best quality and diversity, stable | slow sampling |
| Flow matching | ODE, tractable | regression on velocities | 1–100 passes | simple objective, few steps | same as diffusion, newer |

> **Key idea.** The families differ less in what they can represent than in what they are asked to optimize. GANs optimize a divergence estimated by a critic; diffusion models optimize a denoising regression; VAEs optimize a bound. Those objectives, not the architectures, explain the characteristic look of their samples.

## 2. GAN theory

### 2.1 The minimax game

A **generative adversarial network** (Goodfellow et al., 2014) consists of a **generator** $G_\theta$, which maps noise $z \sim p(z) = \mathcal{N}(0, I)$ to a sample $G_\theta(z)$, and a **discriminator** $D_\phi$, which maps an image to the probability that it came from the data rather than from the generator. They play a two-player game with the value function

$$
V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}\big[\log D(x)\big] + \mathbb{E}_{z \sim p(z)}\big[\log\big(1 - D(G(z))\big)\big].
$$

The discriminator *maximizes* $V$: it wants $D(x) \to 1$ on data and $D(G(z)) \to 0$ on samples, so $V$ is simply the negative binary cross-entropy of a classifier with labels "real" and "fake". The generator *minimizes* $V$: it wants the discriminator to be wrong on its samples. The GAN objective is the saddle-point problem

$$
\min_G \max_D V(D, G).
$$

Note what is *not* here: no reconstruction, no density, no encoder. The only thing the generator ever learns from is the discriminator's opinion.

### 2.2 The optimal discriminator

To understand what the game optimizes, fix the generator and ask what the best possible discriminator looks like. Write $p_g$ for the distribution of $G(z)$; the second expectation is then an expectation over $x \sim p_g$, and

$$
V(D, G) = \int_x \Big[ p_{\text{data}}(x) \log D(x) + p_g(x) \log\big(1 - D(x)\big) \Big] dx .
$$

The integrand can be maximized *pointwise*: at each $x$ we choose $y = D(x) \in (0, 1)$ to maximize $a \log y + b \log(1 - y)$ with $a = p_{\text{data}}(x) \ge 0$ and $b = p_g(x) \ge 0$. Setting the derivative to zero,

$$
\frac{a}{y} - \frac{b}{1 - y} = 0 \quad\Longrightarrow\quad y^* = \frac{a}{a + b},
$$

and the second derivative $-a / y^2 - b / (1 - y)^2 < 0$ confirms a maximum. Therefore the optimal discriminator for a fixed generator is

$$
\boxed{\;D^*_G(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}\;}
$$

— a beautifully interpretable object. It outputs $\tfrac{1}{2}$ wherever the two densities agree, and its deviation from $\tfrac{1}{2}$ measures the local density ratio. A perfect discriminator is a *density-ratio estimator*.

### 2.3 The generator minimizes the Jensen–Shannon divergence

Substituting $D^*_G$ back into the value function gives the quantity the generator actually minimizes,

$$
C(G) = \max_D V(D, G) = \mathbb{E}_{p_{\text{data}}}\left[\log \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}\right] + \mathbb{E}_{p_g}\left[\log \frac{p_g(x)}{p_{\text{data}}(x) + p_g(x)}\right].
$$

Introduce the mixture $m = \tfrac{1}{2}(p_{\text{data}} + p_g)$, so that $\frac{p_{\text{data}}}{p_{\text{data}} + p_g} = \frac{1}{2}\frac{p_{\text{data}}}{m}$ and likewise for $p_g$. Pulling the two factors of $\log\tfrac{1}{2}$ out of the expectations,

$$
C(G) = -\log 4 + \mathrm{KL}\big(p_{\text{data}} \,\|\, m\big) + \mathrm{KL}\big(p_g \,\|\, m\big) = -\log 4 + 2\, \mathrm{JS}\big(p_{\text{data}} \,\|\, p_g\big),
$$

where the **Jensen–Shannon divergence** is defined as $\mathrm{JS}(p \| q) = \tfrac{1}{2}\mathrm{KL}(p \| m) + \tfrac{1}{2}\mathrm{KL}(q \| m)$. It is symmetric, non-negative, bounded by $\log 2$, and zero exactly when $p = q$. Hence the global minimum of the game is $C(G^*) = -\log 4$, attained if and only if $p_g = p_{\text{data}}$, at which point $D^* \equiv \tfrac{1}{2}$. This is the theoretical justification of the adversarial idea: *with an optimal discriminator, training the generator is minimizing a proper divergence between the model and the data*.

The argument also contains the seed of every practical GAN difficulty. The discriminator is never optimal — it is a network updated by a few gradient steps — so the generator minimizes a noisy, lagging estimate of the divergence. And the Jensen–Shannon divergence itself misbehaves when $p_{\text{data}}$ and $p_g$ do not overlap, as Section 4 shows.

### 2.4 The non-saturating loss and the training algorithm

The generator's part of $V$ is $\mathbb{E}_z[\log(1 - D(G(z)))]$. Write the discriminator as a logit, $D = \sigma(u)$. Then $\frac{d}{du}\log(1 - \sigma(u)) = -\sigma(u)$, which is close to zero exactly when the discriminator is confident that the sample is fake, $\sigma(u) \approx 0$ — the situation at the *beginning* of training, when the generator needs gradient the most. This is the **saturating** loss. Goodfellow et al. proposed instead to *maximize* $\log D(G(z))$, whose logit-derivative $\frac{d}{du}\log\sigma(u) = 1 - \sigma(u)$ is largest precisely when the sample is confidently rejected. This **non-saturating** loss,

$$
\mathcal{L}_G = -\mathbb{E}_z\big[\log D(G(z))\big],
$$

has the same fixed point but much healthier gradients, and it is what practically every GAN implementation uses. In code it is the binary cross-entropy of $D(G(z))$ against the label "real". The full algorithm alternates two updates per mini-batch:

1. **Discriminator step.** Sample a real batch and a fake batch $G(z)$ (detached, so no gradient flows into $G$); take one gradient step on $\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1 - D(G(z)))]$.
2. **Generator step.** Sample fresh noise; take one gradient step on $\mathcal{L}_G$, back-propagating through $D$ into $G$ (the discriminator's parameters receive gradients too but are not updated).

Before building image GANs, let us verify the theory numerically on a one-dimensional problem where we know both densities: $p_{\text{data}} = \mathcal{N}(0, 1)$ and a "generator" $p_g = \mathcal{N}(2, 1.5^2)$. We train a small discriminator MLP on samples only and compare it with the analytic $D^*$, and we check the identity $V(D^*, G) = -\log 4 + 2\,\mathrm{JS}$.

In [ ]:
p_data = torch.distributions.Normal(0.0, 1.0)
p_gen = torch.distributions.Normal(2.0, 1.5)

torch.manual_seed(SEED)
D_toy = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
opt = torch.optim.Adam(D_toy.parameters(), lr=1e-2)
for step in range(1500):
    x_real, x_fake = p_data.sample((512, 1)), p_gen.sample((512, 1))
    loss = (F.binary_cross_entropy_with_logits(D_toy(x_real), torch.ones(512, 1))
            + F.binary_cross_entropy_with_logits(D_toy(x_fake), torch.zeros(512, 1)))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

xs = torch.linspace(-5, 7, 1201)[:, None]
with torch.no_grad():
    d_learned = torch.sigmoid(D_toy(xs)).squeeze(1)
pd_, pg_ = p_data.log_prob(xs).exp().squeeze(1), p_gen.log_prob(xs).exp().squeeze(1)
d_star = pd_ / (pd_ + pg_)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(xs.squeeze(1), pd_, label="$p_{data}$"); axes[0].plot(xs.squeeze(1), pg_, label="$p_g$")
axes[0].set_xlabel("x"); axes[0].set_ylabel("density"); axes[0].set_title("The two distributions"); axes[0].legend()
axes[1].plot(xs.squeeze(1), d_star, label="$D^* = p_{data} / (p_{data} + p_g)$", lw=2)
axes[1].plot(xs.squeeze(1), d_learned, "--", label="learned discriminator")
axes[1].set_xlabel("x"); axes[1].set_ylabel("D(x)"); axes[1].set_title("Optimal versus learned discriminator"); axes[1].legend()
plt.tight_layout(); plt.show()

dx = (xs[1, 0] - xs[0, 0]).item()
m_ = 0.5 * (pd_ + pg_)
js = 0.5 * torch.sum(pd_ * torch.log(pd_ / m_)) * dx + 0.5 * torch.sum(pg_ * torch.log(pg_ / m_)) * dx
v_star = torch.sum(pd_ * torch.log(d_star)) * dx + torch.sum(pg_ * torch.log(1 - d_star)) * dx
print(f"max |D_learned - D*| on the plotted range: {(d_learned - d_star).abs().max():.3f}")
print(f"V(D*, G) by numerical integration : {v_star:.4f}")
print(f"-log 4 + 2 JS(p_data || p_g)       : {-math.log(4) + 2 * js:.4f}")
print(f"V(D_learned, G) from the last batch: {-loss.item():.4f}   (JS = {js:.4f} nats, max possible {math.log(2):.4f})")

The learned discriminator tracks the analytic density ratio closely wherever either density is non-negligible (in the far tails it has seen no data and drifts, which is harmless). The numerically integrated value of the game at $D^*$ matches $-\log 4 + 2\,\mathrm{JS}$ to the precision of the grid, and the value achieved by the *learned* discriminator on a fresh batch is slightly lower (up to mini-batch noise), as it must be for a sub-optimal $D$. The JS divergence here is a moderate 0.25 nats; if we moved $p_g$ far to the right it would saturate at $\log 2 \approx 0.69$ — keep that number in mind for Section 4.

## 3. DCGAN on MNIST

### 3.1 Architectures and the DCGAN guidelines

The first GANs used MLPs and produced 28 × 28 digits with visible artefacts. Radford, Metz and Chintala (2016) found a set of architectural rules that made convolutional GANs train reliably, and their **DCGAN** recipe is still the baseline:

- Replace pooling by **strided convolutions** in the discriminator and **transposed convolutions** in the generator, so that the networks learn their own down- and up-sampling.
- Use **BatchNorm** in both networks (except at the generator output and the discriminator input), which keeps activations in range and helps gradient flow.
- Remove fully connected hidden layers; the generator's only dense layer maps $z$ to the first feature map, and the discriminator ends in a single linear unit.
- Use **ReLU** in the generator with a **tanh** output, and **LeakyReLU** (slope 0.2) in the discriminator.
- Initialize all weights from $\mathcal{N}(0, 0.02^2)$, and train with Adam at learning rate $2 \times 10^{-4}$ and $\beta_1 = 0.5$ rather than the default 0.9 — the lower momentum matters, because in a two-player game the target moves and stale momentum causes oscillation.

Our generator maps a 100-dimensional $z$ to a $7 \times 7$ feature map and doubles the resolution twice; the discriminator mirrors it. Both accept an optional class label — appended to $z$ as a one-hot vector in the generator and as constant input channels in the discriminator — which we ignore until the conditional GAN of Section 3.6. The discriminator also has a switch to drop BatchNorm, which the WGAN-GP critic of Section 4 requires.

In [ ]:
class Generator(nn.Module):
    """DCGAN generator: z (+ optional one-hot label) -> 1x28x28 image in [-1, 1]."""

    def __init__(self, z_dim=Z_DIM, width=WIDTH_GAN, n_classes=0):
        super().__init__()
        self.z_dim, self.width, self.n_classes = z_dim, width, n_classes
        self.fc = nn.Sequential(
            nn.Linear(z_dim + n_classes, 2 * width * 7 * 7), nn.BatchNorm1d(2 * width * 7 * 7), nn.ReLU())
        self.net = nn.Sequential(
            nn.ConvTranspose2d(2 * width, width, 4, stride=2, padding=1), nn.BatchNorm2d(width), nn.ReLU(),  # 7 -> 14
            nn.ConvTranspose2d(width, 1, 4, stride=2, padding=1), nn.Tanh(),                                 # 14 -> 28
        )

    def forward(self, z, y=None):
        if self.n_classes > 0:
            z = torch.cat([z, F.one_hot(y, self.n_classes).float()], dim=1)
        return self.net(self.fc(z).view(z.shape[0], 2 * self.width, 7, 7))


class Discriminator(nn.Module):
    """DCGAN discriminator or WGAN critic: 1x28x28 image (+ optional label) -> one real-valued output."""

    def __init__(self, width=WIDTH_GAN, n_classes=0, use_bn=True):
        super().__init__()
        self.n_classes = n_classes
        norm = nn.BatchNorm2d(2 * width) if use_bn else nn.Identity()
        self.net = nn.Sequential(
            nn.Conv2d(1 + n_classes, width, 4, stride=2, padding=1), nn.LeakyReLU(0.2),          # 28 -> 14
            nn.Conv2d(width, 2 * width, 4, stride=2, padding=1), norm, nn.LeakyReLU(0.2),       # 14 -> 7
            nn.Flatten(), nn.Linear(2 * width * 7 * 7, 1),
        )

    def forward(self, x, y=None):
        if self.n_classes > 0:
            y_maps = F.one_hot(y, self.n_classes).float()[:, :, None, None].expand(-1, -1, 28, 28)
            x = torch.cat([x, y_maps], dim=1)
        return self.net(x).squeeze(1)


def init_dcgan(module):
    """DCGAN initialization: N(0, 0.02) for conv/linear weights, N(1, 0.02) for norm scales."""
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(module.weight, 0.0, 0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        nn.init.normal_(module.weight, 1.0, 0.02); nn.init.zeros_(module.bias)

G_probe, D_probe = Generator().to(device), Discriminator().to(device)
with torch.no_grad():
    z_probe = torch.randn(4, Z_DIM, device=device)
    print("G:", tuple(z_probe.shape), "->", tuple(G_probe(z_probe).shape),
          "| D:", tuple(D_probe(G_probe(z_probe)).shape))
print(f"generator parameters {sum(p.numel() for p in G_probe.parameters()):,} | "
      f"discriminator parameters {sum(p.numel() for p in D_probe.parameters()):,}")

### 3.2 The training loop

The loop below implements the alternating algorithm of Section 2.4 and is written once for every GAN variant in this notebook. The two loss functions are passed in as a pair, so the same loop trains the standard GAN (`"ns"`), the Wasserstein critic (`"wgan"`) and, in Exercise 1, the hinge loss. Three further arguments will be needed later: `n_critic` (discriminator updates per generator update), `gp_weight` (the gradient penalty of Section 4) and `real_target` (one-sided label smoothing, Section 3.5). At six evenly spaced points during training, and once more at the final step, the loop records the generator's output on a *fixed* batch of noise, which is the right way to watch a GAN learn: the same $z$ decoded over time shows how each sample evolves.

In [ ]:
def d_loss_ns(d_real, d_fake, real_target=1.0):
    """Standard GAN discriminator loss: BCE with target real_target for data and 0 for samples."""
    return (F.binary_cross_entropy_with_logits(d_real, torch.full_like(d_real, real_target))
            + F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake)))

def g_loss_ns(d_fake):
    """Non-saturating generator loss: -log D(G(z)), i.e. BCE against the label 'real'."""
    return F.binary_cross_entropy_with_logits(d_fake, torch.ones_like(d_fake))

def d_loss_wgan(d_real, d_fake, real_target=None):
    """WGAN critic loss: E[f(fake)] - E[f(real)] (the negative Wasserstein estimate)."""
    return d_fake.mean() - d_real.mean()

def g_loss_wgan(d_fake):
    """WGAN generator loss: -E[f(G(z))]."""
    return -d_fake.mean()

GAN_LOSSES = {"ns": (d_loss_ns, g_loss_ns), "wgan": (d_loss_wgan, g_loss_wgan)}

def gradient_penalty(D, x_real, x_fake, y=None):
    """WGAN-GP penalty: (||grad_x D(x_hat)||_2 - 1)^2 at random interpolates x_hat between real and fake."""
    alpha = torch.rand(x_real.shape[0], 1, 1, 1, device=x_real.device)
    x_hat = (alpha * x_real + (1.0 - alpha) * x_fake).requires_grad_(True)
    grad = torch.autograd.grad(D(x_hat, y).sum(), x_hat, create_graph=True)[0]
    return ((grad.flatten(1).norm(dim=1) - 1.0) ** 2).mean()

@torch.no_grad()
def sample_gan(G, z, y=None):
    """Generate images from given noise (and labels) in eval mode, then restore train mode."""
    was_training = G.training
    G.eval(); out = G(z, y); G.train(was_training)
    return out

def train_gan(G, D, X, epochs, Y=None, loss="ns", lr_g=2e-4, lr_d=2e-4, betas=(0.5, 0.999),
              n_critic=1, gp_weight=0.0, real_target=1.0, batch_size=BATCH_GAN, n_snapshots=6,
              on_g_step=None, verbose=True):
    """Alternating GAN training. Returns per-step losses and sample snapshots on a fixed noise batch.

    loss: "ns", "wgan", or a pair (d_loss_fn, g_loss_fn) with the signatures used above.
    Y: integer labels for a conditional GAN (G and D must have been built with n_classes=10).
    on_g_step(G): optional callback after every generator update (used for the EMA of Exercise 2).
    """
    d_loss_fn, g_loss_fn = GAN_LOSSES[loss] if isinstance(loss, str) else loss
    is_wgan = isinstance(loss, str) and loss == "wgan"
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g, betas=betas)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d, betas=betas)
    conditional = Y is not None
    tensors = [X] if Y is None else [X, Y]
    steps_per_epoch = math.ceil(X.shape[0] / batch_size)
    total_steps = epochs * steps_per_epoch
    snapshot_every = max(1, total_steps // n_snapshots)
    gen = torch.Generator().manual_seed(SEED)
    fixed_z = torch.randn(N_SAMPLE, G.z_dim, generator=gen).to(device)
    fixed_y = (torch.arange(N_SAMPLE, device=device) % 10) if conditional else None
    hist = {"d_loss": [], "g_loss": [], "d_real": [], "d_fake": [], "snapshots": [], "snapshot_steps": []}
    step = 0
    for epoch in range(1, epochs + 1):
        G.train(); D.train(); t0 = time.time()
        ep = {"d_loss": [], "g_loss": [], "d_real": [], "d_fake": []}
        for batch in iterate_batches(tensors, batch_size):
            x_real = batch[0]
            y = batch[1] if conditional else None
            B = x_real.shape[0]
            # --- discriminator / critic update ---
            with torch.no_grad():
                x_fake = G(torch.randn(B, G.z_dim, device=device), y)
            d_real, d_fake = D(x_real, y), D(x_fake, y)
            d_loss = d_loss_fn(d_real, d_fake, real_target)
            if gp_weight > 0:
                d_loss = d_loss + gp_weight * gradient_penalty(D, x_real, x_fake, y)
            opt_d.zero_grad(set_to_none=True); d_loss.backward(); opt_d.step()
            # --- generator update, after every n_critic discriminator updates ---
            if (step + 1) % n_critic == 0:
                g_loss = g_loss_fn(D(G(torch.randn(B, G.z_dim, device=device), y), y))
                opt_g.zero_grad(set_to_none=True); g_loss.backward(); opt_g.step()
                if on_g_step is not None:
                    on_g_step(G)
                hist["g_loss"].append(g_loss.item()); ep["g_loss"].append(g_loss.item())
            hist["d_loss"].append(d_loss.item()); ep["d_loss"].append(d_loss.item())
            hist["d_real"].append(torch.sigmoid(d_real).mean().item()); ep["d_real"].append(hist["d_real"][-1])
            hist["d_fake"].append(torch.sigmoid(d_fake).mean().item()); ep["d_fake"].append(hist["d_fake"][-1])
            step += 1
            if step % snapshot_every == 0 or step == total_steps:
                hist["snapshots"].append(sample_gan(G, fixed_z, fixed_y)); hist["snapshot_steps"].append(step)
        if verbose:
            if is_wgan:
                print(f"epoch {epoch:2d} | critic loss {np.mean(ep['d_loss']):7.3f} (W estimate {-np.mean(ep['d_loss']):6.3f}) "
                      f"| G loss {np.mean(ep['g_loss']):7.3f} | {time.time() - t0:.1f}s")
            else:
                print(f"epoch {epoch:2d} | D loss {np.mean(ep['d_loss']):.3f} | G loss {np.mean(ep['g_loss']):.3f} "
                      f"| D(real) {np.mean(ep['d_real']):.3f} | D(fake) {np.mean(ep['d_fake']):.3f} | {time.time() - t0:.1f}s")
    return hist

def show_snapshots(hist, title):
    """Show the fixed-noise sample grids recorded during training, side by side."""
    n = len(hist["snapshots"])
    fig, axes = plt.subplots(1, n, figsize=(1.9 * n, 2.3))
    for ax, imgs, s in zip(np.atleast_1d(axes), hist["snapshots"], hist["snapshot_steps"]):
        grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=int(math.sqrt(N_SAMPLE)), padding=1, pad_value=0.5)
        ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"step {s}", fontsize=9); ax.axis("off")
    plt.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

### 3.3 Training the DCGAN

Under the `FAST_RUN` budget the generator sees only a few hundred mini-batches, which is enough to go from noise to stroke-like blobs and, usually, to the first recognizable digits; the full budget (20 epochs on all 60,000 images, a few minutes on a GPU) gives clean digits. Watch the printed `D(real)` and `D(fake)` columns: a healthy run keeps both away from the extremes, typically `D(real)` around 0.6–0.8 and `D(fake)` around 0.2–0.4, meaning that the discriminator is ahead but not winning outright.

In [ ]:
torch.manual_seed(SEED)
G_dc, D_dc = Generator().to(device), Discriminator().to(device)
G_dc.apply(init_dcgan); D_dc.apply(init_dcgan)
print("--- DCGAN, non-saturating loss ---")
hist_dc = train_gan(G_dc, D_dc, X_train, EPOCHS_GAN)
show_snapshots(hist_dc, "DCGAN samples from a fixed noise batch during training")

Now the loss curves and the discriminator's average outputs on real and generated images, step by step. Look at them once, and then read the interpretation below before ever using such curves to judge a GAN.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(hist_dc["d_loss"], label="discriminator loss", alpha=0.8)
axes[0].plot(np.linspace(0, len(hist_dc["d_loss"]), len(hist_dc["g_loss"])), hist_dc["g_loss"],
             label="generator loss", alpha=0.8)
axes[0].set_xlabel("discriminator step"); axes[0].set_ylabel("loss"); axes[0].set_title("GAN losses"); axes[0].legend()
axes[1].plot(hist_dc["d_real"], label="D(real)", alpha=0.8); axes[1].plot(hist_dc["d_fake"], label="D(fake)", alpha=0.8)
axes[1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1].set_xlabel("discriminator step"); axes[1].set_ylabel("mean probability"); axes[1].set_ylim(0, 1)
axes[1].set_title("Discriminator outputs"); axes[1].legend()
plt.tight_layout(); plt.show()

The snapshots show the same sixteen (or sixty-four) noise vectors decoded at successive points of training: blobs sharpen into strokes, strokes into digits, and each position on the grid keeps its identity, because the mapping $z \mapsto G(z)$ changes smoothly. The loss curves, on the other hand, tell you almost nothing. Unlike a classifier's loss, a GAN loss is not a measure of progress: it is one player's score in a game against an opponent that is also improving. The generator's loss can *rise* while its samples get better, simply because the discriminator got better faster. The only readings worth taking from these curves are pathological ones — the discriminator loss collapsing to zero with `D(fake)` pinned at 0 means the generator has stopped receiving useful gradient, and violent oscillation means the learning rates are too high. Judge a GAN by its samples (and, for a number, by a feature distance such as FID, see Section 7 of Notebook 11 · Autoencoders and Variational Autoencoders), never by its loss.

> **Common pitfall.** Do not "early-stop" a GAN on its generator loss, and do not compare two GANs by their losses. The value of the game depends on the discriminator, which is different in every run.

### 3.4 Instability and mode collapse

GAN training can fail in two characteristic ways. In **discriminator dominance** the discriminator becomes near-perfect early, its outputs saturate, and the generator's gradient vanishes (the saturation argument of Section 2.4 applies to the non-saturating loss too when $D$ is *very* confident, because $D$'s own gradient with respect to the image becomes tiny). In **mode collapse** the generator maps many different $z$ to nearly the same image: it has found one output that the current discriminator accepts and exploits it; the discriminator then learns to reject that image, the generator jumps to another mode, and the two chase each other around the data distribution without ever covering it. Both failures are made more likely by aggressive optimization — high learning rates and, above all, Adam's default $\beta_1 = 0.9$, which lets momentum carry the players past each other. The next cell deliberately uses such a configuration on the same architecture.

In [ ]:
torch.manual_seed(SEED)
G_bad, D_bad = Generator().to(device), Discriminator().to(device)
G_bad.apply(init_dcgan); D_bad.apply(init_dcgan)
print("--- deliberately unstable configuration: lr = 1e-3 for both players, Adam beta1 = 0.9 ---")
hist_bad = train_gan(G_bad, D_bad, X_train, 1, lr_g=1e-3, lr_d=1e-3, betas=(0.9, 0.999))
show_snapshots(hist_bad, "Unstable configuration: fixed-noise samples during training")

plt.figure(figsize=(7, 3))
plt.plot(hist_bad["d_real"], label="D(real)"); plt.plot(hist_bad["d_fake"], label="D(fake)")
plt.plot(hist_dc["d_fake"][:len(hist_bad["d_fake"])], ":", color="gray", label="D(fake), healthy run")
plt.ylim(0, 1); plt.xlabel("discriminator step"); plt.ylabel("mean probability"); plt.legend()
plt.title("Discriminator outputs in the unstable run"); plt.tight_layout(); plt.show()

Because GAN failure is stochastic, what you see here depends on the seed and the budget, but the typical picture is one of the following: the discriminator wins outright (`D(fake)` drops to nearly 0 and stays there, and the samples stop improving), or the samples in the fixed-noise grid become near-copies of one another (mode collapse), or the outputs oscillate between a few patterns from one snapshot to the next. If your run happens to look healthy, increase both learning rates to $2 \times 10^{-3}$ and rerun; instability is never far away.

### 3.5 Stabilization tricks

A large literature of tricks exists, and it helps to know which problem each one addresses.

- **One-sided label smoothing** (Salimans et al., 2016): train the discriminator with target 0.9 instead of 1 for real images. It stops $D$ from becoming arbitrarily confident on data, which keeps its gradients alive for the generator. Only the *real* side is smoothed; smoothing the fake side would reward the generator for producing images the discriminator rejects.
- **Two-time-scale update rule** (TTUR; Heusel et al., 2017): give the discriminator a higher learning rate than the generator (or several updates per generator update). Theory says the game converges to a local equilibrium if the discriminator is the "faster" player.
- **Spectral normalization** (Miyato et al., 2018): divide each weight matrix of the discriminator by its largest singular value, which constrains the Lipschitz constant of $D$ and smooths its gradients — the same goal as the Wasserstein critic of Section 4, achieved architecturally. `torch.nn.utils.parametrizations.spectral_norm` implements it.
- **Feature matching** (Salimans et al., 2016): train the generator to match the *mean intermediate features* of the discriminator on real and fake batches instead of fooling its final output; it reduces mode collapse by giving the generator a statistic of the whole batch to match.
- **Minibatch discrimination / minibatch standard deviation**: let the discriminator see statistics across the batch, so that a collapsed generator producing identical images is easy to detect.
- **Exponential moving average of the generator weights** (Exercise 2): sample from a slowly updated copy of $G$; it averages out the oscillation of the game and is nearly free.
- **Hinge loss** (Exercise 1), the **R1 gradient penalty** on real data (Mescheder et al., 2018), and **progressive growing / StyleGAN** architectures address stability at high resolution.

> **Try it.** Retrain the DCGAN with `real_target=0.9` (one-sided label smoothing) and with `lr_d=4e-4` (TTUR) and compare the `D(real)`/`D(fake)` traces with the baseline.

### 3.6 Conditional GAN

A **conditional GAN** (Mirza and Osindero, 2014) gives the label $y$ to both players: the generator draws $G(z, y)$ and the discriminator judges the pair $(x, y)$, so that it rejects not only unrealistic images but also realistic images of the *wrong* class. The generator is thereby forced to respect the label, and at sampling time we can ask for any digit. Our classes already support this through `n_classes=10`.

In [ ]:
torch.manual_seed(SEED)
G_c, D_c = Generator(n_classes=10).to(device), Discriminator(n_classes=10).to(device)
G_c.apply(init_dcgan); D_c.apply(init_dcgan)
print("--- conditional DCGAN ---")
hist_c = train_gan(G_c, D_c, X_train, EPOCHS_GAN, Y=y_train)

torch.manual_seed(SEED)
z_grid = torch.randn(8, Z_DIM, device=device).repeat(10, 1)                    # 8 styles, repeated for 10 digits
y_grid = torch.arange(10, device=device).repeat_interleave(8)                  # 0,0,...,0,1,1,...,9
show_images(sample_gan(G_c, z_grid, y_grid), nrow=8, figsize=(5.5, 7),
            title="Conditional GAN: rows = requested digit, columns = shared z")

Each row shows one requested digit, and each column reuses the same noise vector across rows. As in the conditional VAE of Notebook 11, the label controls identity and $z$ controls style, but the GAN samples are markedly sharper: the discriminator penalizes blur as readily as it penalizes a wrong shape, whereas a per-pixel likelihood rewards hedging. Under the `FAST_RUN` budget the rows may be only partly correct — the discriminator needs time to learn what each label should look like — and the full budget produces clean, class-consistent digits.

## 4. Wasserstein GAN

### 4.1 Why the Jensen–Shannon divergence fails on disjoint supports

Section 2.3 showed that an ideal GAN minimizes $\mathrm{JS}(p_{\text{data}} \| p_g)$. Consider the simplest possible learning problem: the data are uniform on the interval $[0, 1]$ and the generator produces a uniform distribution on $[\theta, \theta + 1]$, with a single parameter $\theta$ to learn. Whenever $|\theta| \ge 1$ the two supports are disjoint, and then every term of the JS divergence is at its maximum: $\mathrm{KL}(p \| m) = \log 2$ because $p / m = 2$ wherever $p > 0$, and likewise for $q$, so $\mathrm{JS} = \log 2$ regardless of *how far apart* the intervals are. The divergence is flat, its gradient with respect to $\theta$ is zero, and gradient descent cannot move the generator towards the data. The KL divergence is worse still — infinite. Real image distributions live on low-dimensional manifolds inside pixel space, so at the start of training the data manifold and the generator's manifold essentially never intersect, and an optimal discriminator can separate them perfectly: this is discriminator dominance from Section 3.4, seen from the divergence side.

The **Wasserstein-1 distance** (also called the earth mover's distance) behaves differently. It measures the minimum cost of transporting the mass of $p$ onto $q$, where moving a unit of mass over a distance $d$ costs $d$:

$$
W_1(p, q) = \inf_{\gamma \in \Pi(p, q)} \mathbb{E}_{(x, y) \sim \gamma}\big[\|x - y\|\big],
$$

with $\Pi(p, q)$ the set of joint distributions with marginals $p$ and $q$ (the transport plans). For our two intervals $W_1 = |\theta|$: it grows *linearly* with the gap and its gradient points the generator in the right direction, whether or not the supports overlap. In one dimension it can be computed from the cumulative distribution functions, $W_1(p, q) = \int |F_p(x) - F_q(x)|\, dx$, which is how the next cell evaluates it.

In [ ]:
grid = torch.linspace(-3.0, 4.0, 7001)
dx = (grid[1] - grid[0]).item()

def uniform_pdf(a, b):
    """Density of the uniform distribution on [a, b) evaluated on the grid."""
    return ((grid >= a) & (grid < b)).float() / (b - a)

def js_divergence(p, q):
    """Jensen-Shannon divergence of two densities on the grid (nats)."""
    m = 0.5 * (p + q)
    ratio_p = torch.where(p > 0, p / m.clamp_min(1e-12), torch.ones_like(p))
    ratio_q = torch.where(q > 0, q / m.clamp_min(1e-12), torch.ones_like(q))
    return (0.5 * torch.sum(p * torch.log(ratio_p)) + 0.5 * torch.sum(q * torch.log(ratio_q))) * dx

def wasserstein_1d(p, q):
    """W1 between two densities on the grid, via the integral of |F_p - F_q|."""
    F_p, F_q = torch.cumsum(p, dim=0) * dx, torch.cumsum(q, dim=0) * dx
    return torch.sum((F_p - F_q).abs()) * dx

thetas = np.linspace(-2.5, 2.5, 101)
p_unit = uniform_pdf(0.0, 1.0)
js_vals = [js_divergence(p_unit, uniform_pdf(th, th + 1.0)).item() for th in thetas]
w1_vals = [wasserstein_1d(p_unit, uniform_pdf(th, th + 1.0)).item() for th in thetas]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(thetas, js_vals, label="JS( U[0,1] || U[theta, theta+1] )")
axes[0].axhline(math.log(2), color="gray", ls="--", lw=1, label="log 2")
axes[0].set_xlabel("theta"); axes[0].set_ylabel("nats"); axes[0].set_title("Jensen-Shannon divergence"); axes[0].legend()
axes[1].plot(thetas, w1_vals, color="C1", label="W1( U[0,1], U[theta, theta+1] )")
axes[1].set_xlabel("theta"); axes[1].set_ylabel("distance"); axes[1].set_title("Wasserstein-1 distance"); axes[1].legend()
plt.tight_layout(); plt.show()

For $|\theta| \ge 1$ the Jensen–Shannon divergence sits at $\log 2$ with zero slope, so a generator starting at $\theta = 2$ receives no signal at all; even in the overlapping region its shape is a cusp rather than a smooth bowl. The Wasserstein distance is $|\theta|$ everywhere, with a gradient of constant magnitude pointing at the data. This is the whole argument of Arjovsky, Chintala and Bottou (2017): train the generator to minimize a distance that is *continuous and almost everywhere differentiable* in the generator's parameters, even when the supports are disjoint.

### 4.2 The critic and the gradient penalty

The transport definition of $W_1$ is not something we can optimize directly over neural networks. The **Kantorovich–Rubinstein duality** rewrites it as a maximization over functions,

$$
W_1(p_{\text{data}}, p_g) = \sup_{\|f\|_L \le 1} \; \mathbb{E}_{x \sim p_{\text{data}}}\big[f(x)\big] - \mathbb{E}_{x \sim p_g}\big[f(x)\big],
$$

where the supremum runs over all **1-Lipschitz** functions, $|f(x) - f(y)| \le \|x - y\|$. The Wasserstein GAN replaces the discriminator by a **critic** $f_w$ trained to maximize this difference, and trains the generator to minimize $-\mathbb{E}_z[f_w(G(z))]$. The critic outputs an unbounded real number, not a probability, there is no sigmoid and no cross-entropy, and — a real practical benefit — the critic's loss $\mathbb{E}[f(G(z))] - \mathbb{E}[f(x)]$ is an estimate of $-W_1$ that *does* correlate with sample quality, unlike the GAN losses of Section 3.3.

Everything hinges on the Lipschitz constraint. The original WGAN enforced it crudely by clipping the critic's weights to $[-0.01, 0.01]$, which works but wastes capacity and makes the critic's behaviour depend on the clipping value. **WGAN-GP** (Gulrajani et al., 2017) uses the fact that an optimal critic has gradient norm exactly 1 almost everywhere along the transport paths between $p_{\text{data}}$ and $p_g$, and adds a **gradient penalty** on random interpolates $\hat{x} = \alpha x_{\text{real}} + (1 - \alpha) x_{\text{fake}}$, $\alpha \sim U[0, 1]$:

$$
\mathcal{L}_{\text{critic}} = \mathbb{E}_{z}\big[f_w(G(z))\big] - \mathbb{E}_{x}\big[f_w(x)\big] + \lambda\, \mathbb{E}_{\hat{x}}\Big[\big(\|\nabla_{\hat{x}} f_w(\hat{x})\|_2 - 1\big)^2\Big], \qquad \lambda = 10.
$$

Computing this penalty requires differentiating a gradient, so `torch.autograd.grad(..., create_graph=True)` is used inside the loss — the `gradient_penalty` function of Section 3.2 does exactly that. Two further details of the recipe: the critic must **not** use BatchNorm, because the penalty is defined per input and BatchNorm couples the inputs of a batch (we pass `use_bn=False`), and the critic is updated several times (`n_critic = 5`) per generator update so that it stays close to optimal, with Adam at $\beta_1 = 0$.

In [ ]:
torch.manual_seed(SEED)
G_w, D_w = Generator().to(device), Discriminator(use_bn=False).to(device)
G_w.apply(init_dcgan); D_w.apply(init_dcgan)
print("--- WGAN-GP: 5 critic updates per generator update, gradient penalty 10 ---")
hist_w = train_gan(G_w, D_w, X_train, EPOCHS_WGAN, loss="wgan", lr_g=2e-4, lr_d=2e-4, betas=(0.0, 0.9),
                   n_critic=5, gp_weight=10.0)
show_snapshots(hist_w, "WGAN-GP samples from a fixed noise batch during training")

w_estimate = -np.array(hist_w["d_loss"])
plt.figure(figsize=(7, 3))
plt.plot(w_estimate, alpha=0.5, label="per critic step")
kernel = max(1, len(w_estimate) // 50)
plt.plot(np.convolve(w_estimate, np.ones(kernel) / kernel, mode="valid"), color="C3", label=f"moving average ({kernel} steps)")
plt.xlabel("critic step"); plt.ylabel("E[f(real)] - E[f(fake)]  (minus the penalty)")
plt.title("WGAN-GP: the critic loss estimates the Wasserstein distance"); plt.legend()
plt.tight_layout(); plt.show()

The critic's objective (plotted with its sign flipped, so that larger means "further apart") rises at first, while the critic learns to separate the two distributions, and then decreases as the generator closes the gap — a curve you can actually read, in contrast with the GAN losses. Under the `FAST_RUN` budget the generator performs only a few dozen updates (one per five critic steps), so its samples lag behind the DCGAN's at the same wall-clock time; with the full budget WGAN-GP reaches similar quality and does so with far less sensitivity to the learning rate and the architecture. That robustness, not the final image quality, is the reason Wasserstein-style critics and gradient penalties became standard ingredients of large GANs such as BigGAN and StyleGAN.

> **Going deeper.** Spectral normalization (Section 3.5) enforces a Lipschitz bound on every layer and hence on the whole critic, without the extra backward pass of the gradient penalty; the hinge loss of Exercise 1 is its usual companion. Whether one thinks of these as "Wasserstein" methods or simply as regularized discriminators, the lesson is the same: the discriminator's gradient with respect to its input is the generator's only teacher, and it must be kept smooth.

## 5. Diffusion models: the theory

### 5.1 The forward process and its closed form

A **denoising diffusion probabilistic model** (Sohl-Dickstein et al., 2015; Ho, Jain and Abbeel, 2020) starts by defining a way to *destroy* data. The **forward process** adds a small amount of Gaussian noise at each of $T$ steps, according to a **variance schedule** $\beta_1, \dots, \beta_T \in (0, 1)$:

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\big(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t I\big), \qquad q(x_{1:T} \mid x_0) = \prod_{t=1}^{T} q(x_t \mid x_{t-1}).
$$

The scaling by $\sqrt{1 - \beta_t}$ is chosen so that the variance does not blow up: if $x_{t-1}$ has unit variance per pixel, then so does $x_t$, since $(1 - \beta_t) + \beta_t = 1$. Written as a sample, with $\alpha_t = 1 - \beta_t$,

$$
x_t = \sqrt{\alpha_t}\, x_{t-1} + \sqrt{1 - \alpha_t}\, \epsilon_{t-1}, \qquad \epsilon_{t-1} \sim \mathcal{N}(0, I).
$$

The crucial property is that we can jump to any step in one go. Substituting the expression for $x_{t-1}$,

$$
x_t = \sqrt{\alpha_t \alpha_{t-1}}\, x_{t-2} + \sqrt{\alpha_t (1 - \alpha_{t-1})}\, \epsilon_{t-2} + \sqrt{1 - \alpha_t}\, \epsilon_{t-1}.
$$

The last two terms are independent zero-mean Gaussians, and the sum of two independent Gaussians with variances $\sigma_1^2$ and $\sigma_2^2$ is a Gaussian with variance $\sigma_1^2 + \sigma_2^2$. Here $\alpha_t(1 - \alpha_{t-1}) + (1 - \alpha_t) = 1 - \alpha_t\alpha_{t-1}$, so the two noise terms merge into $\sqrt{1 - \alpha_t\alpha_{t-1}}\, \bar{\epsilon}$. Repeating the substitution down to $x_0$ and writing $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ gives the **closed form** of the forward process:

$$
\boxed{\;q(x_t \mid x_0) = \mathcal{N}\big(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) I\big), \qquad x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I).\;}
$$

Training will sample a random $t$ and produce $x_t$ directly with this formula, never simulating the chain. The schedule must satisfy $\bar{\alpha}_T \approx 0$, so that $x_T$ is indistinguishable from pure noise: then the generative process can *start* from $\mathcal{N}(0, I)$.

### 5.2 Variance schedules

Ho et al. used a **linear** schedule, $\beta_t$ increasing linearly from $10^{-4}$ to $0.02$ over $T = 1000$ steps, which gives $\sum_t \beta_t \approx 10$ and $\bar{\alpha}_T \approx e^{-10} \approx 0$. Because our `FAST_RUN` configuration uses a shorter chain, the code below scales $\beta_T$ by $1000 / T$ so that the total amount of noise is the same for any $T$. Nichol and Dhariwal (2021) observed that the linear schedule destroys the image too quickly — roughly the last fifth of the steps operate on pure noise and can be skipped at sampling time with almost no loss in quality — and proposed the **cosine** schedule, defined directly through

$$
\bar{\alpha}_t = \frac{f(t)}{f(0)}, \qquad f(t) = \cos^2\!\left(\frac{t / T + s}{1 + s} \cdot \frac{\pi}{2}\right), \qquad s = 0.008,
$$

from which the $\beta_t = 1 - \bar{\alpha}_t / \bar{\alpha}_{t-1}$ follow (Exercise 3 implements and trains with it). A schedule is best read through its **signal-to-noise ratio**, $\mathrm{SNR}(t) = \bar{\alpha}_t / (1 - \bar{\alpha}_t)$: the linear schedule's log-SNR falls steeply and then flattens at a very negative value, while the cosine schedule spreads the decrease evenly over $t$. The `DiffusionSchedule` class pre-computes every coefficient we will need, including the posterior variance $\tilde{\beta}_t$ of Section 5.3; note that the code indexes steps as $t = 0, \dots, T - 1$ where the mathematics uses $1, \dots, T$.

In [ ]:
def linear_betas(T, beta_start=1e-4, beta_end=None):
    """Linear schedule of Ho et al. (1e-4 .. 0.02 for T = 1000); beta_end is scaled so that sum(beta) ~ 10 for any T."""
    if beta_end is None:
        beta_end = 0.02 * 1000.0 / T
    return torch.linspace(beta_start, beta_end, T)

def cosine_alpha_bar(t_over_T, s=0.008):
    """alpha_bar(t) of the cosine schedule of Nichol and Dhariwal, for t/T in [0, 1]."""
    f = torch.cos((t_over_T + s) / (1 + s) * math.pi / 2) ** 2
    return f / math.cos(s / (1 + s) * math.pi / 2) ** 2

class DiffusionSchedule:
    """Pre-computed coefficients of a DDPM with T steps (code index t = 0 ... T-1 is the paper's t = 1 ... T)."""

    def __init__(self, betas):
        self.T = betas.shape[0]
        self.betas = betas.to(device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.alpha_bar_prev = torch.cat([torch.ones(1, device=device), self.alpha_bar[:-1]])
        self.sqrt_alpha_bar = self.alpha_bar.sqrt()
        self.sqrt_one_minus_alpha_bar = (1.0 - self.alpha_bar).sqrt()
        self.posterior_var = self.betas * (1.0 - self.alpha_bar_prev) / (1.0 - self.alpha_bar)

    def q_sample(self, x0, t, eps):
        """Closed-form forward process: x_t = sqrt(alpha_bar_t) x0 + sqrt(1 - alpha_bar_t) eps, t of shape (B,)."""
        return (self.sqrt_alpha_bar[t][:, None, None, None] * x0
                + self.sqrt_one_minus_alpha_bar[t][:, None, None, None] * eps)

sched = DiffusionSchedule(linear_betas(T_DIFF))
print(f"T = {sched.T}: beta from {sched.betas[0]:.2e} to {sched.betas[-1]:.3f}, "
      f"alpha_bar_T = {sched.alpha_bar[-1]:.2e}")

t_frac = (torch.arange(T_DIFF, device=device).float() + 1) / T_DIFF   # the paper's t / T for code index 0 ... T-1
ab_cos = cosine_alpha_bar(t_frac)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(t_frac.cpu(), sched.alpha_bar.cpu(), label="linear")
axes[0].plot(t_frac.cpu(), ab_cos.cpu(), label="cosine")
axes[0].set_xlabel("t / T"); axes[0].set_ylabel(r"$\bar{\alpha}_t$"); axes[0].set_title("Signal fraction"); axes[0].legend()
for ab, lab in [(sched.alpha_bar, "linear"), (ab_cos, "cosine")]:
    snr = ab / (1 - ab).clamp_min(1e-12)
    axes[1].plot(t_frac.cpu(), torch.log(snr.clamp_min(1e-12)).cpu(), label=lab)
axes[1].set_xlabel("t / T"); axes[1].set_ylabel("log SNR"); axes[1].set_title("Log signal-to-noise ratio"); axes[1].legend()
plt.tight_layout(); plt.show()

# What the forward process does to an image at a few noise levels
torch.manual_seed(SEED)
x0_demo = X_diff[:1].expand(6, -1, -1, -1)
t_demo = torch.tensor([0.0, 0.1, 0.25, 0.5, 0.75, 1.0])
t_idx = (t_demo * (T_DIFF - 1)).long().to(device)
x_noised = sched.q_sample(x0_demo, t_idx, torch.randn_like(x0_demo))
show_images(x_noised, nrow=6, figsize=(7, 1.6),
            title="Forward process (linear schedule) at t/T = " + ", ".join(f"{v:.2f}" for v in t_demo.tolist()))

By $t / T = 0.5$ the digit is barely visible under the linear schedule and at $0.75$ it is gone: the log-SNR plot confirms that the second half of the linear chain operates at signal-to-noise ratios far below anything a human could decode, while the cosine curve keeps some signal until much later. Whichever schedule is used, the *end* of the chain must be pure noise, and the printed $\bar{\alpha}_T$ confirms that it is.

### 5.3 The reverse process and why it is Gaussian

Generation runs the chain backwards: start from $x_T \sim \mathcal{N}(0, I)$ and sample $x_{t-1}$ from $q(x_{t-1} \mid x_t)$ repeatedly. The true reversal $q(x_{t-1} \mid x_t)$ depends on the unknown data distribution and is intractable in general — but for *small* $\beta_t$ it has the same functional form as the forward step. Feller (1949) showed this for diffusion processes: reversing a Gaussian diffusion with small step size yields another Gaussian diffusion. Intuitively, if $x_t$ is very close to $x_{t-1}$, then over that tiny neighbourhood the data density is approximately constant and Bayes' rule turns the Gaussian forward kernel into a Gaussian backward kernel. This is why $T$ must be large: the approximation gets better as each step gets smaller. The model therefore *parameterizes* the reverse transitions as Gaussians whose means (and possibly variances) are computed by a network,

$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\big(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 I\big), \qquad p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t),
$$

with $\sigma_t^2$ fixed to either $\beta_t$ or $\tilde{\beta}_t$ defined below (Ho et al. found both to work).

There is one reverse distribution we *can* write down exactly: the posterior of $x_{t-1}$ given $x_t$ **and the clean image** $x_0$. By Bayes' rule, $q(x_{t-1} \mid x_t, x_0) \propto q(x_t \mid x_{t-1})\, q(x_{t-1} \mid x_0)$, a product of two Gaussians in $x_{t-1}$, hence Gaussian. Completing the square gives

$$
q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\big(x_{t-1};\; \tilde{\mu}_t(x_t, x_0),\; \tilde{\beta}_t I\big), \quad
\tilde{\mu}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1 - \bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\,(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}\, x_t, \quad
\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\, \beta_t .
$$

This tractable posterior is the target the network will learn to imitate.

### 5.4 From the ELBO to the $\epsilon$-prediction objective

A diffusion model is a latent-variable model with latents $x_{1:T}$, so the variational machinery of Notebook 11 · Autoencoders and Variational Autoencoders applies: with $q(x_{1:T} \mid x_0)$ as the (fixed, non-learned) approximate posterior,

$$
-\log p_\theta(x_0) \;\le\; \mathbb{E}_q\left[-\log p(x_T) - \sum_{t=1}^{T} \log \frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}\right] =: L.
$$

Rewriting the forward kernels with Bayes' rule, $q(x_t \mid x_{t-1}) = q(x_{t-1} \mid x_t, x_0)\, q(x_t \mid x_0) / q(x_{t-1} \mid x_0)$, makes the sum telescope and yields the celebrated decomposition of Sohl-Dickstein et al.,

$$
L = \underbrace{\mathrm{KL}\big(q(x_T \mid x_0)\,\|\,p(x_T)\big)}_{L_T \text{ (no parameters)}}
+ \sum_{t=2}^{T} \underbrace{\mathrm{KL}\big(q(x_{t-1} \mid x_t, x_0)\,\|\,p_\theta(x_{t-1} \mid x_t)\big)}_{L_{t-1}}
+ \underbrace{\big(-\log p_\theta(x_0 \mid x_1)\big)}_{L_0}.
$$

Every $L_{t-1}$ is a KL divergence between two Gaussians with the same covariance, which is simply a scaled squared distance between their means:

$$
L_{t-1} = \frac{1}{2\sigma_t^2}\, \mathbb{E}_q\Big[\big\|\tilde{\mu}_t(x_t, x_0) - \mu_\theta(x_t, t)\big\|^2\Big] + \text{const}.
$$

The network could predict $\tilde{\mu}_t$ directly, but a change of variables gives a much better-conditioned target. From the closed form, $x_0 = \big(x_t - \sqrt{1 - \bar{\alpha}_t}\,\epsilon\big) / \sqrt{\bar{\alpha}_t}$; substituting this into $\tilde{\mu}_t$ and simplifying the coefficients (using $\bar{\alpha}_t = \alpha_t \bar{\alpha}_{t-1}$) gives

$$
\tilde{\mu}_t(x_t, x_0) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon\right).
$$

So the posterior mean is $x_t$ minus a known multiple of *the noise that was added*. It is natural to let the network predict that noise, $\epsilon_\theta(x_t, t)$, and to define the model mean with the same formula,

$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right).
$$

Then the difference of means is proportional to the difference of noises, and

$$
L_{t-1} = \frac{\beta_t^2}{2\sigma_t^2\, \alpha_t\, (1 - \bar{\alpha}_t)}\; \mathbb{E}_{x_0, \epsilon}\Big[\big\|\epsilon - \epsilon_\theta\big(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\,\epsilon,\; t\big)\big\|^2\Big] + \text{const}.
$$

Ho et al.'s decisive simplification was to *drop the weights* in front of the expectation, which down-weights the small-$t$ terms (where the true weight is large and the task is nearly trivial) and up-weights the large-$t$ terms where the network must learn global structure. Sampling $t$ uniformly, the training objective becomes the **simple loss**

$$
\boxed{\;L_{\text{simple}}(\theta) = \mathbb{E}_{t \sim U\{1..T\},\; x_0 \sim p_{\text{data}},\; \epsilon \sim \mathcal{N}(0, I)}\Big[\big\|\epsilon - \epsilon_\theta\big(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\,\epsilon,\; t\big)\big\|^2\Big].\;}
$$

Everything about training a diffusion model is contained in this line: pick an image, pick a step, add the corresponding amount of noise, ask the network which noise was added, and regress. It is a plain supervised regression problem — no adversary, no posterior collapse, no sampling inside the loss — which is why diffusion training is so stable.

### 5.5 The connection to score matching

Because $q(x_t \mid x_0)$ is Gaussian, its score is available in closed form: $\nabla_{x_t} \log q(x_t \mid x_0) = -\frac{x_t - \sqrt{\bar{\alpha}_t} x_0}{1 - \bar{\alpha}_t} = -\frac{\epsilon}{\sqrt{1 - \bar{\alpha}_t}}$. Predicting $\epsilon$ is therefore the same as predicting the score of the noised data distribution, up to a known scale, and $L_{\text{simple}}$ is a **denoising score-matching** objective (Vincent, 2011) summed over noise levels — the very identity mentioned for denoising autoencoders in Section 2.5 of Notebook 11 · Autoencoders and Variational Autoencoders. Song et al. (2021) unified the two views: the forward process is a stochastic differential equation, the reverse process is another SDE whose drift involves the score, and there is also a deterministic *probability-flow ODE* with the same marginals. DDIM (Section 6.5) is a discretization of that ODE, and flow matching (Section 7) learns a related ODE directly.

## 6. DDPM on MNIST

### 6.1 Sinusoidal time embedding

The network must know the noise level it is operating at, because the optimal denoiser differs completely between $t = 10$ (remove a faint speckle) and $t = 900$ (guess the global shape). The standard mechanism is the sinusoidal embedding of Notebook 09 · Attention and Transformers, applied to the integer $t$: a vector of sines and cosines at geometrically spaced frequencies, passed through a small MLP and *added* to the feature maps inside every residual block.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Map integer time steps (B,) to (B, dim) vectors of sines and cosines at geometric frequencies."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

emb_demo = SinusoidalTimeEmbedding(64)(torch.arange(T_DIFF))
plt.figure(figsize=(7, 3))
plt.imshow(emb_demo.T.numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="value"); plt.grid(False)
plt.xlabel("time step t"); plt.ylabel("embedding dimension"); plt.title("Sinusoidal time embedding (64-d)")
plt.tight_layout(); plt.show()

### 6.2 A small time-conditioned U-Net

The **U-Net** (Ronneberger et al., 2015; adapted to diffusion by Ho et al.) is the workhorse architecture of image diffusion. Its encoder halves the resolution twice ($28 \to 14 \to 7$) and doubles the channels once ($C \to 2C$), a bottleneck processes the coarsest map, and the decoder upsamples back, **concatenating** the encoder's feature map of the same resolution at every level (the *skip connections* that give the U its shape). The skips are what make denoising work: fine detail comes straight across from the encoder, while global structure is decided in the bottleneck. Every stage is a **residual block** with GroupNorm, SiLU activations and a projection of the time embedding added to the features; a single self-attention layer at the $7 \times 7$ resolution lets every position see every other, which helps global coherence at negligible cost. The class also accepts an optional label embedding, added to the time embedding, with an extra "null" class index reserved for classifier-free guidance (Section 6.6). Notebook 07 · Transfer Learning and Modern Vision built a U-Net for segmentation; this one differs only in the conditioning.

In [ ]:
class ResBlock(nn.Module):
    """GroupNorm-SiLU-Conv twice, with the time embedding projected and added after the first conv."""

    def __init__(self, c_in, c_out, t_dim):
        super().__init__()
        self.norm1, self.conv1 = nn.GroupNorm(8, c_in), nn.Conv2d(c_in, c_out, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, c_out)
        self.norm2, self.conv2 = nn.GroupNorm(8, c_out), nn.Conv2d(c_out, c_out, 3, padding=1)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SelfAttention2d(nn.Module):
    """Multi-head self-attention over the spatial positions of a feature map, with a residual connection."""

    def __init__(self, channels, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, 3 * channels, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)   # start as the identity

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(self.norm(x)).reshape(B, 3, self.n_heads, C // self.n_heads, H * W)
        q, k, v = (u.transpose(-1, -2) for u in qkv.unbind(dim=1))          # each (B, heads, HW, head_dim)
        out = F.scaled_dot_product_attention(q, k, v)                        # (B, heads, HW, head_dim)
        out = out.transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(out)


class UNet(nn.Module):
    """Small time-conditioned U-Net for 28x28 images: 28 -> 14 -> 7 -> 14 -> 28, predicting epsilon."""

    def __init__(self, base=WIDTH_UNET, n_classes=0):
        super().__init__()
        t_dim = 4 * base
        self.n_classes = n_classes
        self.time_embed = nn.Sequential(SinusoidalTimeEmbedding(t_dim), nn.Linear(t_dim, t_dim), nn.SiLU(),
                                        nn.Linear(t_dim, t_dim))
        if n_classes > 0:
            self.class_embed = nn.Embedding(n_classes + 1, t_dim)          # index n_classes = "no label"
        C = base
        self.in_conv = nn.Conv2d(1, C, 3, padding=1)
        self.down1 = ResBlock(C, C, t_dim)                                   # 28x28, C channels
        self.pool1 = nn.Conv2d(C, C, 4, stride=2, padding=1)                 # 28 -> 14
        self.down2 = ResBlock(C, 2 * C, t_dim)                               # 14x14, 2C channels
        self.pool2 = nn.Conv2d(2 * C, 2 * C, 4, stride=2, padding=1)         # 14 -> 7
        self.mid1 = ResBlock(2 * C, 2 * C, t_dim)
        self.mid_attn = SelfAttention2d(2 * C)
        self.mid2 = ResBlock(2 * C, 2 * C, t_dim)
        self.up2 = ResBlock(4 * C, 2 * C, t_dim)                             # 14x14, after concat with down2
        self.up1 = ResBlock(3 * C, C, t_dim)                                 # 28x28, after concat with down1
        self.out = nn.Sequential(nn.GroupNorm(8, C), nn.SiLU(), nn.Conv2d(C, 1, 3, padding=1))

    def forward(self, x, t, y=None):
        emb = self.time_embed(t)
        if self.n_classes > 0:
            if y is None:
                y = torch.full((x.shape[0],), self.n_classes, device=x.device, dtype=torch.long)
            emb = emb + self.class_embed(y)
        h1 = self.down1(self.in_conv(x), emb)                                # (B, C, 28, 28)
        h2 = self.down2(self.pool1(h1), emb)                                 # (B, 2C, 14, 14)
        h = self.pool2(h2)                                                   # (B, 2C, 7, 7)
        h = self.mid2(self.mid_attn(self.mid1(h, emb)), emb)
        h = F.interpolate(h, scale_factor=2.0, mode="nearest")               # 7 -> 14
        h = self.up2(torch.cat([h, h2], dim=1), emb)                         # (B, 2C, 14, 14)
        h = F.interpolate(h, scale_factor=2.0, mode="nearest")               # 14 -> 28
        h = self.up1(torch.cat([h, h1], dim=1), emb)                         # (B, C, 28, 28)
        return self.out(h)

unet_probe = UNet().to(device)
with torch.no_grad():
    out = unet_probe(X_diff[:4], torch.tensor([0, 10, 100, T_DIFF - 1], device=device))
print("U-Net:", tuple(X_diff[:4].shape), "->", tuple(out.shape),
      f"| parameters {sum(p.numel() for p in unet_probe.parameters()):,}")

### 6.3 Training

The training loop is the simple loss of Section 5.4, literally: sample $t$, $\epsilon$, form $x_t$ with `q_sample`, predict, regress. We also record the loss separately for ten bins of $t$, which reveals how the difficulty of the regression varies with the noise level. The optional `p_uncond` argument randomly replaces labels by the null class, which is how the classifier-free guidance model of Section 6.6 will be trained.

In [ ]:
def train_ddpm(model, sched, X, epochs, Y=None, p_uncond=0.0, lr=2e-4, batch_size=BATCH_DIFF, verbose=True):
    """Train an epsilon-prediction diffusion model with L_simple. Returns per-step losses and losses by t-bin."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tensors = [X] if Y is None else [X, Y]
    n_bins = 10
    loss_by_t = torch.zeros(n_bins, device=device)
    count_by_t = torch.zeros(n_bins, device=device)
    hist = {"loss": []}
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0; n_seen = 0
        for batch in iterate_batches(tensors, batch_size):
            x0 = batch[0]
            y = batch[1] if Y is not None else None
            B = x0.shape[0]
            t = torch.randint(0, sched.T, (B,), device=device)
            eps = torch.randn_like(x0)
            x_t = sched.q_sample(x0, t, eps)
            if y is not None and p_uncond > 0:
                drop = torch.rand(B, device=device) < p_uncond
                y = torch.where(drop, torch.full_like(y, model.n_classes), y)
            per_sample = (model(x_t, t, y) - eps).pow(2).flatten(1).mean(dim=1)
            loss = per_sample.mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            hist["loss"].append(loss.item()); running += loss.item() * B; n_seen += B
            bins = (t * n_bins) // sched.T
            loss_by_t.index_add_(0, bins, per_sample.detach()); count_by_t.index_add_(0, bins, torch.ones_like(per_sample))
        if verbose:
            print(f"epoch {epoch:2d} | train loss {running / n_seen:.4f} | {time.time() - t0:.1f}s")
    hist["loss_by_t"] = (loss_by_t / count_by_t.clamp_min(1.0)).cpu().numpy()
    return hist

torch.manual_seed(SEED)
unet = UNet().to(device)
print("--- DDPM, unconditional ---")
hist_ddpm = train_ddpm(unet, sched, X_diff, EPOCHS_DIFF)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(hist_ddpm["loss"], alpha=0.8); axes[0].set_xlabel("step"); axes[0].set_ylabel("MSE on epsilon")
axes[0].set_title("Training loss (L_simple)")
axes[1].bar(np.arange(10) / 10 + 0.05, hist_ddpm["loss_by_t"], width=0.09)
axes[1].set_xlabel("t / T (bin)"); axes[1].set_ylabel("MSE on epsilon"); axes[1].set_title("Loss by noise level (all steps)")
plt.tight_layout(); plt.show()

The loss falls smoothly and monotonically — the picture of a well-posed regression problem, and a relief after the GAN curves. The per-bin plot shows where the difficulty lies: at *large* $t$ the input is almost pure noise and the network can predict it almost perfectly (the output is nearly the input), so the loss is small; at *small* $t$ the noise is a faint perturbation of a clean digit and extracting it exactly is hard, so the loss is largest. Even with the uniform weights of $L_{\text{simple}}$ the low-noise end therefore dominates the summed loss; under the true ELBO weights of Section 5.4, which are largest at small $t$, it would dominate far more, and that is exactly why Ho et al. dropped them — the large-$t$ terms, where the network has to learn global structure, would otherwise receive almost no gradient.

### 6.4 Ancestral sampling

Sampling reverses the chain with the learned mean and a fixed variance. Starting from $x_T \sim \mathcal{N}(0, I)$, for $t = T, \dots, 1$:

$$
x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right) + \sigma_t\, \xi, \qquad \xi \sim \mathcal{N}(0, I),
$$

with no noise added at the last step. This is **ancestral sampling** from the model $p_\theta(x_{0:T})$; we use $\sigma_t^2 = \tilde{\beta}_t$. The `predict_eps` helper below wraps the network call and will later add classifier-free guidance; `keep_every` records intermediate states so that we can watch the image emerge from the noise.

In [ ]:
def predict_eps(model, x, t, y=None, guidance=1.0):
    """epsilon_theta(x_t, t), with classifier-free guidance when the model is conditional and y is given."""
    if y is None or model.n_classes == 0 or guidance == 1.0:
        return model(x, t, y)
    y_null = torch.full_like(y, model.n_classes)
    eps_both = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([y, y_null]))
    eps_cond, eps_uncond = eps_both.chunk(2, dim=0)
    return eps_uncond + guidance * (eps_cond - eps_uncond)

@torch.no_grad()
def ddpm_sample(model, sched, n, y=None, guidance=1.0, keep_every=None, seed=SEED, x_init=None):
    """Ancestral DDPM sampling. Returns final images in [-1, 1] and a list of (t, x_t) snapshots."""
    model.eval()
    if x_init is None:
        torch.manual_seed(seed)
        x = torch.randn(n, 1, 28, 28, device=device)
    else:
        x = x_init.clone()
    frames = []
    for t in reversed(range(sched.T)):
        if keep_every is not None and t % keep_every == keep_every - 1:
            frames.append((t, x.clamp(-1, 1).clone()))
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        eps = predict_eps(model, x, tt, y, guidance)
        mean = (x - sched.betas[t] / sched.sqrt_one_minus_alpha_bar[t] * eps) / sched.alphas[t].sqrt()
        if t > 0:
            x = mean + sched.posterior_var[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    x = x.clamp(-1, 1)
    if keep_every is not None:
        frames.append((0, x.clone()))
    return x, frames

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

sync(); t0 = time.time()
samples_ddpm, _ = ddpm_sample(unet, sched, N_SAMPLE)
sync(); t_ddpm = time.time() - t0
show_images(samples_ddpm, nrow=int(math.sqrt(N_SAMPLE)), figsize=(5, 5),
            title=f"DDPM ancestral samples ({sched.T} steps, {t_ddpm:.1f}s for {N_SAMPLE} images)")

Under the `FAST_RUN` budget the network has seen only a few thousand noisy images, so expect digit-like blobs with the right kind of strokes rather than clean digits; the full budget (20 epochs on all of MNIST, a few minutes on a GPU) gives crisp, diverse digits, generally cleaner than the DCGAN's at the same resolution. Note the cost: one image requires $T$ network evaluations, versus one for a GAN or VAE. The next cell shows *how* the samples arise, by recording the state at regular intervals during the reverse chain.

In [ ]:
def show_trajectory(frames, title):
    """Rows: samples; columns: the recorded (t, x_t) snapshots of the reverse process."""
    n = frames[0][1].shape[0]
    imgs = torch.stack([x for _, x in frames], dim=1).reshape(-1, 1, 28, 28)       # (n * n_frames, 1, 28, 28)
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=len(frames), padding=1, pad_value=0.5)
    plt.figure(figsize=(1.1 * len(frames), 1.1 * n))
    plt.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1); plt.axis("off")
    plt.title(title + "\nt = " + ", ".join(str(t) for t, _ in frames), fontsize=9)
    plt.tight_layout(); plt.show()

_, frames = ddpm_sample(unet, sched, 6, keep_every=sched.T // 8, seed=SEED + 1)
show_trajectory(frames, "Denoising trajectory of the ancestral sampler (left: pure noise, right: sample)")

Reading each row from left to right: for most of the chain the state looks like noise to the eye, then a faint blob appears, its global shape is fixed within a short window of steps, and the last part of the chain sharpens strokes and cleans the background. This ordering — coarse structure first, detail last — is the visual counterpart of the signal-to-noise curve: high-frequency detail has low SNR and is decided late. It also hints at where the computation is wasted, which is what DDIM exploits.

### 6.5 DDIM: deterministic sampling with fewer steps

Song, Meng and Ermon (2021) noticed that the training objective depends only on the marginals $q(x_t \mid x_0)$, not on the Markov chain that produced them. One can therefore define a *different* family of forward processes with the same marginals and a reverse process that may be deterministic, and sample from the same trained $\epsilon_\theta$ along a *subsequence* of time steps $\tau_1 < \dots < \tau_S$. Given $x_{\tau_i}$, the model first estimates the clean image,

$$
\hat{x}_0 = \frac{x_{\tau_i} - \sqrt{1 - \bar{\alpha}_{\tau_i}}\; \epsilon_\theta(x_{\tau_i}, \tau_i)}{\sqrt{\bar{\alpha}_{\tau_i}}},
$$

and then re-noises that estimate to the *previous* level with the same predicted noise:

$$
x_{\tau_{i-1}} = \sqrt{\bar{\alpha}_{\tau_{i-1}}}\; \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{\tau_{i-1}} - \sigma_{\tau_i}^2}\; \epsilon_\theta(x_{\tau_i}, \tau_i) + \sigma_{\tau_i}\, \xi.
$$

With $\sigma = 0$ (the **DDIM** case, $\eta = 0$) the update is deterministic: the sample is a fixed function of the initial noise, so the same $x_T$ always yields the same image, interpolations in $x_T$ are meaningful, and — because the update is the Euler step of the probability-flow ODE — a few dozen steps give good samples. The parameter $\eta \in [0, 1]$ interpolates towards DDPM-like stochasticity through $\sigma_{\tau_i} = \eta \sqrt{(1 - \bar{\alpha}_{\tau_{i-1}}) / (1 - \bar{\alpha}_{\tau_i})}\sqrt{1 - \bar{\alpha}_{\tau_i} / \bar{\alpha}_{\tau_{i-1}}}$ (Exercise 4). We compare 5, 10, 20 and 50 DDIM steps against full ancestral sampling, all started from the *same* noise.

In [ ]:
@torch.no_grad()
def ddim_sample(model, sched, n, steps, eta=0.0, y=None, guidance=1.0, seed=SEED, x_init=None):
    """DDIM sampling along `steps` evenly spaced time steps; eta = 0 is deterministic."""
    model.eval()
    if x_init is None:
        torch.manual_seed(seed)
        x = torch.randn(n, 1, 28, 28, device=device)
    else:
        x = x_init.clone()
    taus = torch.linspace(sched.T - 1, 0, steps).round().long().tolist()          # descending subsequence
    for i, t in enumerate(taus):
        ab_t = sched.alpha_bar[t]
        ab_prev = sched.alpha_bar[taus[i + 1]] if i + 1 < len(taus) else torch.tensor(1.0, device=device)
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        eps = predict_eps(model, x, tt, y, guidance)
        x0_hat = (x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()
        sigma = eta * ((1 - ab_prev) / (1 - ab_t) * (1 - ab_t / ab_prev)).clamp_min(0).sqrt()
        x = ab_prev.sqrt() * x0_hat + (1 - ab_prev - sigma ** 2).clamp_min(0).sqrt() * eps
        if eta > 0 and i + 1 < len(taus):
            x = x + sigma * torch.randn_like(x)
    return x.clamp(-1, 1)

torch.manual_seed(SEED)
x_T = torch.randn(8, 1, 28, 28, device=device)
rows, labels = [], []
for steps in [5, 10, 20, 50]:
    sync(); t0 = time.time()
    rows.append(ddim_sample(unet, sched, 8, steps, x_init=x_T)); sync()
    labels.append(f"DDIM {steps:3d} steps ({time.time() - t0:5.2f}s)")
sync(); t0 = time.time()
rows.append(ddpm_sample(unet, sched, 8, x_init=x_T)[0]); sync()
labels.append(f"DDPM {sched.T:3d} steps ({time.time() - t0:5.2f}s)")

fig, axes = plt.subplots(len(rows), 1, figsize=(7, 1.2 * len(rows)))
for ax, imgs, lab in zip(axes, rows, labels):
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=8, padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=8)
plt.suptitle("Same initial noise, different samplers (time for 8 images)", fontsize=10)
plt.tight_layout(); plt.show()

Two things stand out. First, the wall-clock time scales with the number of network evaluations, so 20 DDIM steps are roughly $T/20$ times faster than ancestral sampling — a factor of 50 at $T = 1000$. Second, the DDIM rows are *consistent* with each other: because the sampler is deterministic, the same starting noise produces the same digit at 10, 20 and 50 steps, with more steps only refining the strokes, whereas the stochastic DDPM row from the same noise produces a different set of digits. With too few steps (5) the Euler discretization is coarse and the samples degrade — how coarse one can go is the subject of a whole literature of higher-order samplers (DPM-Solver, Heun/EDM samplers) and of distillation into one-step generators.

### 6.6 Classifier-free guidance

To generate a *requested* digit we need a class-conditional model $\epsilon_\theta(x_t, t, y)$. Simply training with the label works, but the resulting samples often respect the class only weakly, because the model has no incentive to make the class *obvious*. The original remedy, **classifier guidance** (Dhariwal and Nichol, 2021), pushes the sampling trajectory along the gradient of a separately trained noise-aware classifier, $\nabla_{x_t}\log p_\phi(y \mid x_t)$ (Exercise 5). **Classifier-free guidance** (Ho and Salimans, 2022) achieves the same effect without any classifier. Train one network to be both conditional and unconditional by replacing the label with a null token $\varnothing$ in a fraction $p_{\text{uncond}} \approx 0.1$ of the training examples, then at sampling time extrapolate *away* from the unconditional prediction:

$$
\tilde{\epsilon}_\theta(x_t, t, y) = \epsilon_\theta(x_t, t, \varnothing) + s\,\big(\epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \varnothing)\big).
$$

With guidance scale $s = 1$ this is the plain conditional model, $s = 0$ the unconditional one, and $s > 1$ amplifies whatever distinguishes the conditional prediction from the unconditional one — i.e. the features that make the image look like class $y$. Through the score view of Section 5.5, the difference of the two noise predictions is proportional to $\nabla_{x_t}\log p(y \mid x_t)$ (by Bayes' rule, $\log p(x_t \mid y) - \log p(x_t) = \log p(y \mid x_t) + \text{const}$), so classifier-free guidance *is* classifier guidance with an implicit classifier derived from the diffusion model itself. It costs two network evaluations per step, which `predict_eps` batches into one call. Large $s$ sharpens class identity and image quality but reduces diversity and eventually produces saturated, over-simplified images: guidance trades fidelity against diversity, exactly like the truncation trick of GANs.

In [ ]:
torch.manual_seed(SEED)
unet_cond = UNet(n_classes=10).to(device)
print("--- class-conditional DDPM with 10% label dropout (for classifier-free guidance) ---")
hist_cond = train_ddpm(unet_cond, sched, X_diff, EPOCHS_DIFF, Y=y_diff, p_uncond=0.1)

torch.manual_seed(SEED)
x_T10 = torch.randn(10, 1, 28, 28, device=device)
y_10 = torch.arange(10, device=device)
GUIDANCE_SCALES = [0.0, 1.0, 3.0, 7.0]
DDIM_STEPS_CFG = 50
rows = [ddim_sample(unet_cond, sched, 10, DDIM_STEPS_CFG, y=y_10, guidance=s, x_init=x_T10) for s in GUIDANCE_SCALES]

fig, axes = plt.subplots(len(rows), 1, figsize=(7, 1.2 * len(rows)))
for ax, imgs, s in zip(axes, rows, GUIDANCE_SCALES):
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=10, padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_ylabel(f"s = {s:.0f}", rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle(f"Classifier-free guidance, digits 0-9 requested, DDIM {DDIM_STEPS_CFG} steps, same initial noise", fontsize=10)
plt.tight_layout(); plt.show()

Row by row: at $s = 0$ the label is ignored (the null token) and the digits are whatever the noise happens to produce; at $s = 1$ the conditional model mostly follows the request; at $s = 3$ the requested digits are unmistakable and the strokes bolder; at $s = 7$ they become caricatures — thick, high-contrast, stripped of stylistic variety. Under the `FAST_RUN` budget the effect is attenuated because the conditional model is barely trained, but the increase in class consistency with $s$ should still be visible. This single scalar is what "guidance scale" or "CFG" means in every modern text-to-image system, where the class label is replaced by a text embedding and the same extrapolation controls how literally the prompt is followed.

## 7. The modern landscape and flow matching

### 7.1 Latent diffusion and text conditioning

The DDPM we trained operates on $28 \times 28$ pixels. Scaling the same recipe to $1024 \times 1024$ colour images would multiply the cost of every one of the hundreds of network evaluations per sample by more than a thousand. **Latent diffusion** (Rombach et al., 2022) avoids this by running the diffusion process in the compressed latent space of a pretrained autoencoder — the low-KL VAE of Section 8.2 of Notebook 11 · Autoencoders and Variational Autoencoders — where a $512 \times 512 \times 3$ image is a $64 \times 64 \times 4$ tensor. The U-Net then denoises latents and the VAE decoder renders the final latent once. Stable Diffusion, and most open image generators since, are latent diffusion models.

**Text conditioning** generalizes the class label of Section 6.6. A frozen text encoder — the CLIP text tower in Stable Diffusion 1 and 2, T5 and larger encoders later — maps the prompt to a sequence of embeddings, and the U-Net attends to that sequence through **cross-attention** layers inserted at several resolutions: image positions are the queries, text tokens the keys and values (Notebook 09 · Attention and Transformers). Classifier-free guidance applies unchanged, with an empty prompt as the null condition, and the guidance scale is the "CFG" slider of every text-to-image interface. More recent systems replace the convolutional U-Net by a Transformer over latent patches (**diffusion transformers**, Peebles and Xie, 2023), which scale more predictably with compute.

### 7.2 Fewer steps: distillation, consistency models and flow matching

The remaining weakness of diffusion is sampling cost, and three ideas attack it. **Progressive distillation** trains a student to reproduce two teacher steps in one, halving the step count repeatedly. **Consistency models** (Song et al., 2023) train a network to map any point on a trajectory directly to its endpoint, which permits one- or two-step generation. **Flow matching** (Lipman et al., 2023; Liu, Gong and Liu, 2023 under the name rectified flow) rethinks the training objective so that the trajectories are as straight as possible from the start, and it is now the default formulation in state-of-the-art systems (Stable Diffusion 3, Flux, and the video generators). Its derivation is short enough to give in full and to test on a toy problem.

Consider a noise sample $x_0 \sim \mathcal{N}(0, I)$ and a data sample $x_1 \sim p_{\text{data}}$, and connect them by the straight line

$$
x_t = (1 - t)\, x_0 + t\, x_1, \qquad t \in [0, 1],
$$

which moves with the constant velocity $u_t = \dot{x}_t = x_1 - x_0$. If we could learn a velocity field $v_\theta(x, t)$ such that the ODE $\dot{x} = v_\theta(x, t)$ transports the Gaussian at $t = 0$ to the data at $t = 1$, sampling would be numerical integration of that ODE. The **conditional flow matching** objective regresses the network onto the velocity of the straight line through each pair:

$$
\boxed{\;\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim U[0,1],\; x_0 \sim \mathcal{N}(0, I),\; x_1 \sim p_{\text{data}}}\Big[\big\| v_\theta\big((1 - t)x_0 + t x_1,\; t\big) - (x_1 - x_0)\big\|^2\Big].\;}
$$

The central theorem of Lipman et al. states that the minimizer of this objective is the *marginal* velocity field $\mathbb{E}[x_1 - x_0 \mid x_t = x, t]$, and that integrating this field does transport the noise distribution exactly onto the data distribution, even though the individual straight lines cross each other and the learned field must therefore bend. The parallel with $L_{\text{simple}}$ is exact: pick a data point, pick a time, corrupt, regress. The differences are that time is continuous, the "corruption" is linear interpolation instead of the variance-preserving Gaussian path, and the target is a velocity instead of a noise. The diffusion forward process is a special case of the same framework with a curved path, so a DDPM can be re-read as a flow model with a particular schedule; the straight path simply makes the ODE easier to integrate with few steps.

We train a flow-matching model on a two-dimensional swiss-roll distribution, where the vector field and the trajectories can be drawn.

In [ ]:
def swiss_roll_2d(n, noise=0.4, seed=SEED):
    """A 2-D swiss roll (the spiral projection of sklearn's 3-D swiss roll), scaled to roughly [-2.5, 2.5]."""
    pts, _ = make_swiss_roll(n_samples=n, noise=noise, random_state=seed)
    return torch.as_tensor(pts[:, [0, 2]] / 6.0, dtype=torch.float32, device=device)

class VelocityNet(nn.Module):
    """MLP velocity field v(x, t) for 2-D points; t is appended to x as a third input."""

    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

data_fm = swiss_roll_2d(8_000)
torch.manual_seed(SEED)
v_net = VelocityNet().to(device)
opt = torch.optim.Adam(v_net.parameters(), lr=2e-3)
FM_STEPS = 1_500 if FAST_RUN else 6_000
fm_losses = []
t0 = time.time()
for step in range(1, FM_STEPS + 1):
    x1 = data_fm[torch.randint(0, data_fm.shape[0], (512,), device=device)]
    x0 = torch.randn_like(x1)
    t = torch.rand(512, 1, device=device)
    x_t = (1 - t) * x0 + t * x1
    loss = F.mse_loss(v_net(x_t, t), x1 - x0)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    fm_losses.append(loss.item())
    if step % (FM_STEPS // 5) == 0:
        print(f"step {step:5d} | flow-matching loss {np.mean(fm_losses[-100:]):.4f} | {time.time() - t0:.1f}s")

@torch.no_grad()
def flow_sample(v_net, n, n_steps=100, seed=SEED):
    """Integrate dx/dt = v(x, t) from t = 0 (noise) to t = 1 (data) with Euler steps; returns x_1 and the path."""
    torch.manual_seed(seed)
    x = torch.randn(n, 2, device=device)
    path = [x.clone()]
    dt = 1.0 / n_steps
    for i in range(n_steps):
        x = x + dt * v_net(x, torch.full((n, 1), i * dt, device=device))
        path.append(x.clone())
    return x, torch.stack(path)                                   # (n_steps + 1, n, 2)

x_gen, path = flow_sample(v_net, 2_000)
x_gen_np, path_np, data_np = x_gen.cpu().numpy(), path.cpu().numpy(), data_fm[:2_000].cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=3, alpha=0.4, label="data")
axes[0].scatter(x_gen_np[:, 0], x_gen_np[:, 1], s=3, alpha=0.4, label="flow-matching samples (100 Euler steps)")
axes[0].set_title("Swiss roll: data and samples"); axes[0].legend(loc="upper left"); axes[0].set_aspect("equal")
for j in range(40):
    axes[1].plot(path_np[:, j, 0], path_np[:, j, 1], lw=0.8, alpha=0.7, color="C1")
axes[1].scatter(path_np[0, :40, 0], path_np[0, :40, 1], s=12, color="gray", label="x at t = 0 (noise)")
axes[1].scatter(path_np[-1, :40, 0], path_np[-1, :40, 1], s=12, color="C3", label="x at t = 1 (sample)")
axes[1].scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.15, color="C0")
axes[1].set_title("Forty trajectories of the learned ODE"); axes[1].legend(loc="upper left"); axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()

The samples land on the spiral, including its inner turns, from a plain regression objective and a tiny MLP. The trajectories are nearly straight for most of their length and bend only near the end, where they must decide which arm of the spiral to join — those are the points where the straight conditional paths of different training pairs cross, and the marginal field has to compromise. The straighter the trajectories, the fewer Euler steps are needed; *rectified flow* reduces the remaining curvature by retraining on pairs $(x_0, x_1)$ generated by the model itself ("reflow"), which is how few-step generators are obtained.

> **Try it.** Reduce `n_steps` in `flow_sample` to 10 and then to 2. Compare with the DDIM experiment of Section 6.5: which model degrades more gracefully, and why does the straight-line path help?

### 7.3 Evaluation and compute in practice

The evaluation toolkit of Section 7 of Notebook 11 · Autoencoders and Variational Autoencoders applies to all of these models: FID and its feature-space relatives, precision/recall for fidelity versus coverage, and for text-to-image systems a **CLIP score** (agreement between the image and the prompt in CLIP's joint embedding space) plus, increasingly, human preference ratings. Likelihood-based comparison is available for diffusion and flow models through the probability-flow ODE, but is rarely reported because it correlates poorly with perceived quality. On the cost side, training a modern text-to-image model takes on the order of $10^5$ GPU-hours and hundreds of millions of image–text pairs; sampling takes tens of network evaluations at $64 \times 64 \times 4$ latent resolution, which is why step-count reduction (Section 7.2) and guidance-distillation (folding the two CFG evaluations into one) receive so much engineering attention.

## 8. Comparison of generative model families

| Criterion | VAE (Notebook 11 · Autoencoders and Variational Autoencoders) | GAN | Diffusion / flow |
|---|---|---|---|
| Objective | ELBO (bound on log-likelihood) | adversarial game (JS or Wasserstein) | denoising / velocity regression (weighted bound) |
| Sample quality | blurry, all modes covered | sharp, may miss modes | sharp and diverse; state of the art |
| Likelihood | lower bound available | none | bound available; exact via the flow ODE, rarely used |
| Sampling cost | 1 network pass | 1 network pass | 10–1000 passes (fewer with DDIM, distillation, flows) |
| Training stability | very stable | fragile (collapse, oscillation, hyperparameter-sensitive) | very stable, plain regression |
| Mode coverage | full (likelihood forces it) | partial (mode collapse) | full |
| Latent space | explicit, smooth, encoder available | implicit; no encoder | $x_T$ acts as a latent with DDIM; no encoder (inversion possible) |
| Conditioning / control | conditional VAE | conditional GAN, projection discriminators | classifier / classifier-free guidance, cross-attention |
| Typical use today | compressors for latent diffusion, representation learning, anomaly detection | fast one-step generation, super-resolution, adversarial losses inside other models | images, video, audio, molecules; the default for high-fidelity generation |

The historical arc is worth stating plainly. VAEs gave a principled but blurry generator; GANs gave sharpness at the price of a fragile training game; diffusion recovered stability by turning generation into regression, at the price of slow sampling; and flow matching and distillation are now removing that price. Each family also survives *inside* the others: the VAE as the compressor of latent diffusion, the adversarial loss as a sharpening term in autoencoders and distilled samplers, and the denoising objective as the training signal for almost everything else.

## Exercises

**Exercise 1 (warm-up): the hinge loss.** Implement the hinge GAN losses (Lim and Ye, 2017; Miyato et al., 2018), $\mathcal{L}_D = \mathbb{E}[\max(0, 1 - D(x))] + \mathbb{E}[\max(0, 1 + D(G(z)))]$ and $\mathcal{L}_G = -\mathbb{E}[D(G(z))]$, as a pair of functions with the signatures used by `train_gan`, train a DCGAN with them under the same budget as Section 3.3, and compare the samples. What does the margin do to the discriminator's incentives compared with the cross-entropy loss?

**Solution to Exercise 1**

In [ ]:
def d_loss_hinge(d_real, d_fake, real_target=None):
    """Hinge discriminator loss: real outputs are pushed above +1, fake outputs below -1, with no reward beyond the margin."""
    return F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake).mean()

def g_loss_hinge(d_fake):
    """Hinge generator loss: raise the discriminator's output on samples (linear, never saturates)."""
    return -d_fake.mean()

EPOCHS_EX = 1 if FAST_RUN else 10
torch.manual_seed(SEED)
G_h, D_h = Generator().to(device), Discriminator().to(device)
G_h.apply(init_dcgan); D_h.apply(init_dcgan)
print("--- DCGAN with the hinge loss ---")
hist_h = train_gan(G_h, D_h, X_train, EPOCHS_EX, loss=(d_loss_hinge, g_loss_hinge))
show_snapshots(hist_h, "Hinge-loss DCGAN samples from a fixed noise batch during training")

With the hinge loss the discriminator receives no gradient from real examples it already scores above $+1$ or fakes it already scores below $-1$: once an example is "confidently classified" it stops contributing, so the discriminator cannot run away to arbitrarily large margins the way a cross-entropy discriminator can, and its gradient with respect to the input stays informative for the generator. The generator loss is linear in $D(G(z))$ and never saturates. (The `D(real)`/`D(fake)` columns printed during training are sigmoids of the raw margins, so they are only qualitatively comparable with the standard GAN.) Combined with spectral normalization this pair is the standard recipe of SAGAN and BigGAN.

**Exercise 2: an exponential moving average of the generator.** Implement an `EMA` helper that keeps a frozen copy of the generator whose parameters are updated as $\theta_{\text{ema}} \leftarrow \beta\, \theta_{\text{ema}} + (1 - \beta)\, \theta$ after every generator step, using the `on_g_step` callback of `train_gan`. Train a DCGAN with it and compare samples from the live generator and from the EMA copy on the same noise. Why does averaging help in an adversarial game specifically?

**Solution to Exercise 2**

In [ ]:
class EMA:
    """Exponential moving average of a model's parameters, kept in a frozen deep copy (buffers are mirrored)."""

    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for p_ema, p in zip(self.shadow.parameters(), model.parameters()):
            p_ema.mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)
        for b_ema, b in zip(self.shadow.buffers(), model.buffers()):
            b_ema.copy_(b)

torch.manual_seed(SEED)
G_e, D_e = Generator().to(device), Discriminator().to(device)
G_e.apply(init_dcgan); D_e.apply(init_dcgan)
ema = EMA(G_e, decay=0.99 if FAST_RUN else 0.999)
print("--- DCGAN with an EMA copy of the generator ---")
hist_e = train_gan(G_e, D_e, X_train, EPOCHS_EX, on_g_step=ema.update)

torch.manual_seed(SEED)
z_cmp = torch.randn(N_SAMPLE, Z_DIM, device=device)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, G_model, lab in [(axes[0], G_e, "live generator"), (axes[1], ema.shadow, "EMA generator")]:
    grid = make_grid(((sample_gan(G_model, z_cmp).cpu() + 1) / 2).clamp(0, 1), nrow=int(math.sqrt(N_SAMPLE)),
                     padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(lab)
plt.tight_layout(); plt.show()

The live generator's parameters at any given step reflect its most recent reaction to the discriminator, which is itself reacting to the generator: the two chase each other around the equilibrium and the parameters oscillate. The EMA is a low-pass filter on that oscillation, so its samples are typically smoother and more consistent than those of any single iterate — most visibly on the full budget, where the live generator's samples may flicker between snapshots while the EMA's evolve slowly. The same trick is used for diffusion models (an EMA of the U-Net is what one samples from in every DDPM implementation of note) for a different reason: it averages the noise of stochastic gradient descent on a very noisy regression target.

**Exercise 3: the cosine schedule.** Implement `cosine_betas(T)` that derives the per-step $\beta_t$ from the cosine $\bar{\alpha}_t$ of Section 5.2 (clipping each $\beta_t$ at 0.999), build a `DiffusionSchedule` from it and verify that its cumulative $\bar{\alpha}_t$ reproduces the closed form. Show a digit noised at the same $t$ under both schedules, train a DDPM with the cosine schedule under the same budget as Section 6.3, and compare its samples with the linear-schedule model's.

**Solution to Exercise 3**

In [ ]:
def cosine_betas(T, s=0.008, max_beta=0.999):
    """Per-step betas of the cosine schedule: beta_t = 1 - alpha_bar(t) / alpha_bar(t-1), clipped for stability."""
    t_over_T = torch.arange(T + 1, dtype=torch.float64) / T
    ab = cosine_alpha_bar(t_over_T, s)
    betas = 1.0 - ab[1:] / ab[:-1]
    return betas.clamp(max=max_beta).float()

sched_cos = DiffusionSchedule(cosine_betas(T_DIFF))
ab_closed = cosine_alpha_bar((torch.arange(T_DIFF, device=device).float() + 1) / T_DIFF)
print(f"max |alpha_bar (cumprod) - alpha_bar (closed form)| = {(sched_cos.alpha_bar - ab_closed).abs().max():.2e}")
print(f"cosine: beta from {sched_cos.betas[0]:.2e} to {sched_cos.betas[-1]:.3f}, alpha_bar_T = {sched_cos.alpha_bar[-1]:.2e}")

torch.manual_seed(SEED)
eps_demo = torch.randn_like(x0_demo)
show_images(torch.cat([sched.q_sample(x0_demo, t_idx, eps_demo), sched_cos.q_sample(x0_demo, t_idx, eps_demo)]),
            nrow=6, figsize=(7, 2.8), title="Same digit and noise at t/T = 0, 0.1, 0.25, 0.5, 0.75, 1: linear (top) vs cosine (bottom)")

torch.manual_seed(SEED)
unet_cos = UNet().to(device)
print("--- DDPM with the cosine schedule ---")
hist_cos = train_ddpm(unet_cos, sched_cos, X_diff, EPOCHS_DIFF)
samples_cos, _ = ddpm_sample(unet_cos, sched_cos, N_SAMPLE)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, imgs, lab in [(axes[0], samples_ddpm, "linear schedule"), (axes[1], samples_cos, "cosine schedule")]:
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=int(math.sqrt(N_SAMPLE)), padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(lab)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3))
plt.bar(np.arange(10) / 10 + 0.03, hist_ddpm["loss_by_t"], width=0.04, label="linear")
plt.bar(np.arange(10) / 10 + 0.07, hist_cos["loss_by_t"], width=0.04, label="cosine")
plt.xlabel("t / T (bin)"); plt.ylabel("MSE on epsilon"); plt.title("Loss by noise level under the two schedules"); plt.legend()
plt.tight_layout(); plt.show()

The cumulative product of the derived $\beta_t$ reproduces the closed-form $\bar{\alpha}_t$ to floating-point precision (apart from the very last step, where clipping at 0.999 intervenes). The noised digits show the practical difference: at $t / T = 0.5$ and $0.75$ the cosine schedule still leaves a visible digit, so more of the chain is spent at informative noise levels, and the loss-by-bin profile is flatter. On MNIST at this budget the sample quality of the two models is similar: the cosine schedule was proposed for low-resolution images ($32 \times 32$ and $64 \times 64$ in the paper), where the linear schedule wastes the most steps, but MNIST's simple, high-contrast digits are forgiving of either schedule, so the difference shows up in the loss profile rather than in the samples. The analysis tools carry over directly to harder datasets.

**Exercise 4: DDIM inversion and consistency.** (a) Using the `eta` argument of `ddim_sample`, generate images from the same initial noise with 10 and with 50 steps for $\eta \in \{0, 1\}$ and measure the mean absolute difference between the two step counts. (b) Implement **DDIM inversion**: run the deterministic update *forwards in time* (from $t = 0$ to $t = T$, using the predicted $\epsilon$ at each step to re-noise the estimated $\hat{x}_0$ to the next level) to map a real test digit to a latent $x_T$, then sample from that $x_T$ with DDIM and measure the reconstruction error. Why does this only work for $\eta = 0$?

**Solution to Exercise 4**

In [ ]:
torch.manual_seed(SEED)
x_T8 = torch.randn(8, 1, 28, 28, device=device)
rows, labels = [], []
for eta in [0.0, 1.0]:
    a = ddim_sample(unet, sched, 8, 10, eta=eta, x_init=x_T8)
    b = ddim_sample(unet, sched, 8, 50, eta=eta, x_init=x_T8)
    rows += [a, b]; labels += [f"eta={eta:.0f}, 10 steps", f"eta={eta:.0f}, 50 steps"]
    print(f"eta = {eta:.0f}: mean |x(10 steps) - x(50 steps)| = {(a - b).abs().mean().item():.4f}")

@torch.no_grad()
def ddim_invert(model, sched, x0, steps, y=None):
    """Deterministic DDIM run forwards (t increasing): map images to the latent noise x_T that regenerates them."""
    model.eval()
    taus = torch.linspace(0, sched.T - 1, steps).round().long().tolist()          # ascending
    x, n = x0.clone(), x0.shape[0]
    for i in range(len(taus) - 1):
        t, t_next = taus[i], taus[i + 1]
        ab_t, ab_next = sched.alpha_bar[t], sched.alpha_bar[t_next]
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        eps = predict_eps(model, x, tt, y)
        x0_hat = (x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()
        x = ab_next.sqrt() * x0_hat + (1 - ab_next).sqrt() * eps
    return x

INV_STEPS = 50
x_real8 = X_test[:8]
x_latent = ddim_invert(unet, sched, x_real8, INV_STEPS)
x_rec = ddim_sample(unet, sched, 8, INV_STEPS, eta=0.0, x_init=x_latent)
print(f"DDIM inversion round trip ({INV_STEPS} steps): mean |x_rec - x_real| = {(x_rec - x_real8).abs().mean().item():.4f} "
      f"(pixel range is [-1, 1]); latent std {x_latent.std().item():.3f} (prior: 1.000)")
rows += [x_real8, x_latent.clamp(-1, 1), x_rec]; labels += ["real digits", "inverted x_T", "reconstruction"]

fig, axes = plt.subplots(len(rows), 1, figsize=(7, 1.1 * len(rows)))
for ax, imgs, lab in zip(axes, rows, labels):
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=8, padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=8)
plt.tight_layout(); plt.show()

(a) With $\eta = 0$ the 10-step and 50-step samples differ only in fine detail, because both integrate the same ODE from the same initial condition; with $\eta = 1$ fresh noise is injected at every step, the two runs follow different random paths, and they end at different digits, so the mean absolute difference is several times larger. (b) Inversion exploits the same determinism: the DDIM update is (approximately) invertible, so running it forwards yields a latent that regenerates the input, with an error that shrinks as the number of steps grows and the model improves — on the `FAST_RUN` model the reconstruction is rough, on the full model it is close. The inverted latents have roughly unit standard deviation, i.e. they look like prior samples, which is what makes latent-space editing of *real* images possible in diffusion models (interpolate two inverted latents, or apply guidance during regeneration). A stochastic sampler has no such inverse: the noise it injects is not recoverable.

**Exercise 5 (challenge): classifier guidance.** Train a *noise-conditioned* classifier $p_\phi(y \mid x_t, t)$ on noised MNIST images (a small CNN that receives the time embedding), then implement the classifier-guided sampler of Dhariwal and Nichol (2021) for the *unconditional* DDPM of Section 6.3: at every step replace the predicted noise by $\tilde{\epsilon} = \epsilon_\theta(x_t, t) - \sqrt{1 - \bar{\alpha}_t}\; s\, \nabla_{x_t} \log p_\phi(y \mid x_t, t)$. Generate the digits 0–9 with $s = 0$ and $s = 3$ and use the classifier at $t = 0$ to judge how often the requested digit was produced.

**Solution to Exercise 5**

The derivation is one line of Bayes' rule in score form. The score of the class-conditional noised distribution is $\nabla_{x_t}\log p(x_t \mid y) = \nabla_{x_t}\log p(x_t) + \nabla_{x_t}\log p(y \mid x_t)$, and Section 5.5 identified $\nabla_{x_t}\log p(x_t) = -\epsilon_\theta / \sqrt{1 - \bar{\alpha}_t}$. Converting the guided score back to a noise prediction gives $\tilde{\epsilon} = \epsilon_\theta - \sqrt{1 - \bar{\alpha}_t}\,\nabla_{x_t}\log p_\phi(y \mid x_t)$, and the scale $s$ sharpens the implicit conditional distribution $p(y \mid x_t)^s$. The classifier must be trained on *noised* inputs at all levels, because during sampling it is evaluated on $x_t$, not on clean images.

In [ ]:
class NoisyClassifier(nn.Module):
    """A small CNN classifier p(y | x_t, t) whose first feature map is shifted by a time embedding."""

    def __init__(self, width=32, t_dim=64):
        super().__init__()
        self.time_embed = nn.Sequential(SinusoidalTimeEmbedding(t_dim), nn.Linear(t_dim, t_dim), nn.SiLU())
        self.conv1 = nn.Conv2d(1, width, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, width)
        self.conv2 = nn.Conv2d(width, 2 * width, 3, padding=1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(2 * width * 7 * 7, 128), nn.SiLU(), nn.Linear(128, 10))

    def forward(self, x, t):
        h = F.silu(self.conv1(x) + self.t_proj(self.time_embed(t))[:, :, None, None])
        h = F.max_pool2d(h, 2)                                   # 28 -> 14
        h = F.max_pool2d(F.silu(self.conv2(h)), 2)               # 14 -> 7
        return self.head(h)

torch.manual_seed(SEED)
noisy_clf = NoisyClassifier().to(device)
opt = torch.optim.Adam(noisy_clf.parameters(), lr=1e-3)
EPOCHS_NCLF = 1 if FAST_RUN else 5
for epoch in range(1, EPOCHS_NCLF + 1):
    noisy_clf.train(); t0 = time.time()
    for xb, yb in iterate_batches([X_diff, y_diff], BATCH_DIFF):
        t = torch.randint(0, sched.T, (xb.shape[0],), device=device)
        x_t = sched.q_sample(xb, t, torch.randn_like(xb))
        loss = F.cross_entropy(noisy_clf(x_t, t), yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    noisy_clf.eval()
    with torch.no_grad():
        t0s = torch.zeros(1000, device=device, dtype=torch.long)
        acc_clean = (noisy_clf(X_test[:1000], t0s).argmax(dim=1) == y_test[:1000]).float().mean().item()
        t_mid = torch.full((1000,), sched.T // 2, device=device, dtype=torch.long)
        x_mid = sched.q_sample(X_test[:1000], t_mid, torch.randn_like(X_test[:1000]))
        acc_mid = (noisy_clf(x_mid, t_mid).argmax(dim=1) == y_test[:1000]).float().mean().item()
    print(f"epoch {epoch} | test accuracy at t=0: {acc_clean:.3f} | at t=T/2: {acc_mid:.3f} | {time.time() - t0:.1f}s")

def classifier_guided_sample(model, clf, sched, y, scale, seed=SEED):
    """Ancestral sampling from an unconditional DDPM, steered by the gradient of log p_phi(y | x_t, t)."""
    model.eval(); clf.eval()
    torch.manual_seed(seed)
    n = y.shape[0]
    x = torch.randn(n, 1, 28, 28, device=device)
    for t in reversed(range(sched.T)):
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        with torch.no_grad():
            eps = model(x, tt)
        with torch.enable_grad():
            x_in = x.detach().requires_grad_(True)
            log_p = F.log_softmax(clf(x_in, tt), dim=1).gather(1, y[:, None]).sum()
            grad = torch.autograd.grad(log_p, x_in)[0]
        eps = eps - sched.sqrt_one_minus_alpha_bar[t] * scale * grad
        mean = (x - sched.betas[t] / sched.sqrt_one_minus_alpha_bar[t] * eps) / sched.alphas[t].sqrt()
        if t > 0:
            x = mean + sched.posterior_var[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x.detach().clamp(-1, 1)

y_req = torch.arange(10, device=device)
rows, labels = [], []
for s in [0.0, 3.0]:
    imgs = classifier_guided_sample(unet, noisy_clf, sched, y_req, s)
    with torch.no_grad():
        pred = noisy_clf(imgs, torch.zeros(10, device=device, dtype=torch.long)).argmax(dim=1)
    rows.append(imgs); labels.append(f"s = {s:.0f}  ({(pred == y_req).float().mean().item() * 100:.0f}% as requested)")

fig, axes = plt.subplots(len(rows), 1, figsize=(7, 1.2 * len(rows)))
for ax, imgs, lab in zip(axes, rows, labels):
    grid = make_grid(((imgs.cpu() + 1) / 2).clamp(0, 1), nrow=10, padding=1, pad_value=0.5)
    ax.imshow(grid[0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=8)
plt.suptitle("Classifier guidance of the unconditional DDPM: digits 0-9 requested", fontsize=10)
plt.tight_layout(); plt.show()

With $s = 0$ the sampler is the plain unconditional DDPM and the digits are whatever the noise dictates, so the "as requested" rate is around chance; with $s = 3$ the classifier's gradient steers each trajectory towards the requested class and the rate rises sharply (on the full budget to nearly 100 %). The classifier's accuracy at $t = T/2$, printed during training, shows why a *noise-aware* classifier is essential: a clean-image classifier would be evaluated far outside its training distribution for most of the chain and would return meaningless gradients. Compared with classifier-free guidance, this method needs a second network and a backward pass at every step, and its guidance signal is only as good as the classifier's gradients on noisy inputs — which is why the classifier-free variant won in practice; but the derivation makes the mechanism of guidance transparent, and the same gradient-based steering works with any differentiable scoring function (a CLIP similarity, a reward model, a physical constraint).

## Summary

- Generative families differ mainly in their objectives: VAEs bound the likelihood, GANs play a divergence-minimizing game through a critic, and diffusion and flow models solve a denoising or velocity regression that bounds the likelihood in a weighted way.
- For a fixed generator the optimal discriminator is the density ratio $p_{\text{data}} / (p_{\text{data}} + p_g)$, and with it the generator minimizes $2\,\mathrm{JS}(p_{\text{data}} \| p_g) - \log 4$; the non-saturating loss keeps gradients alive early in training.
- DCGAN's architectural rules (strided and transposed convolutions, BatchNorm, LeakyReLU, Adam with $\beta_1 = 0.5$) make GANs trainable; loss curves are not progress measures, and mode collapse and discriminator dominance are the characteristic failures, addressed by label smoothing, two-time-scale updates, spectral normalization, EMA and hinge losses.
- The Jensen–Shannon divergence is flat on disjoint supports; the Wasserstein-1 distance is not, and WGAN-GP estimates it with a 1-Lipschitz critic enforced by a gradient penalty, giving a critic loss that tracks sample quality.
- The diffusion forward process has the closed form $x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t}\epsilon$; the reverse process is Gaussian for small steps; the ELBO decomposes into per-step KLs, and predicting $\epsilon$ with uniform weights gives the simple, stable training objective.
- A time-conditioned U-Net trained by this regression generates by ancestral sampling over $T$ steps; DDIM reinterprets the model as an ODE and samples deterministically in a few dozen steps with consistent outputs.
- Classifier-free guidance trains one network with label dropout and extrapolates from the unconditional to the conditional prediction; its scale trades class fidelity against diversity and is the "CFG" of text-to-image systems.
- Flow matching regresses a velocity field onto straight-line displacements between noise and data, is a close cousin of diffusion with a straighter path, and underlies the fastest current generators; latent diffusion moves all of this into a VAE's latent space.

## Further reading

- Goodfellow, I. et al., 2014, *Generative Adversarial Nets*, NeurIPS — the original paper, including the optimal-discriminator and JS derivations.
- Radford, A., Metz, L. and Chintala, S., 2016, *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*, ICLR — DCGAN.
- Salimans, T. et al., 2016, *Improved Techniques for Training GANs*, NeurIPS; Miyato, T. et al., 2018, *Spectral Normalization for Generative Adversarial Networks*, ICLR.
- Arjovsky, M., Chintala, S. and Bottou, L., 2017, *Wasserstein GAN*, ICML; Gulrajani, I. et al., 2017, *Improved Training of Wasserstein GANs*, NeurIPS.
- Sohl-Dickstein, J. et al., 2015, *Deep Unsupervised Learning using Nonequilibrium Thermodynamics*, ICML; Ho, J., Jain, A. and Abbeel, P., 2020, *Denoising Diffusion Probabilistic Models*, NeurIPS.
- Nichol, A. and Dhariwal, P., 2021, *Improved Denoising Diffusion Probabilistic Models*, ICML; Dhariwal, P. and Nichol, A., 2021, *Diffusion Models Beat GANs on Image Synthesis*, NeurIPS.
- Song, J., Meng, C. and Ermon, S., 2021, *Denoising Diffusion Implicit Models*, ICLR; Song, Y. et al., 2021, *Score-Based Generative Modeling through Stochastic Differential Equations*, ICLR.
- Ho, J. and Salimans, T., 2022, *Classifier-Free Diffusion Guidance*, arXiv:2207.12598 (first presented at the NeurIPS 2021 Workshop on Deep Generative Models and Downstream Applications).
- Rombach, R. et al., 2022, *High-Resolution Image Synthesis with Latent Diffusion Models*, CVPR; Peebles, W. and Xie, S., 2023, *Scalable Diffusion Models with Transformers*, ICCV.
- Lipman, Y. et al., 2023, *Flow Matching for Generative Modeling*, ICLR; Liu, X., Gong, C. and Liu, Q., 2023, *Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow*, ICLR.
- Bishop, C. M. and Bishop, H., 2024, *Deep Learning: Foundations and Concepts*, Springer, chapters on GANs and diffusion; Goodfellow, I., Bengio, Y. and Courville, A., 2016, *Deep Learning*, MIT Press, chapter 20; and the PyTorch documentation, https://docs.pytorch.org/docs/stable/ (`torch.autograd.grad`, `F.scaled_dot_product_attention`, `nn.GroupNorm`).